# DeepSearch — Local & Free
Uses **Ollama** (local LLM) + **DuckDuckGo** (free search) + **BeautifulSoup** (web scraping).

**Flow:**
1. Generate targeted sub-queries for your topic
2. Search the web via DuckDuckGo
3. Scrape & chunk page content
4. Rank & filter relevant chunks
5. Synthesize a final deep report

## 1. Setup & Config

In [1]:
# !pip install ddgs 
# !pip install requests beautifulsoup4 ollama yt-dlp -q
# !pip show psycopg2-binary 2>/dev/null | head -2; pg_isready 2>/dev/null || echo "pg_isready not found"; psql --version 2>/dev/null || echo "psql not found"

In [2]:
import ollama
import requests
import yt_dlp
from bs4 import BeautifulSoup
from ddgs import DDGS
import time
import re
from IPython.display import display, Markdown

# --- CONFIG ---
MODEL = "llama3.2:1b"        # change to phi3 or phi if preferred
MAX_SEARCH_RESULTS = 6       # URLs per sub-query
MAX_SUB_QUERIES = 3          # sub-queries to generate
MAX_CHUNK_CHARS = 1500       # chars per scraped chunk
TOP_CHUNKS = 8               # top chunks for final synthesis
REQUEST_TIMEOUT = 10         # HTTP timeout in seconds
YT_MAX_RESULTS = 6           # YouTube videos to return

print(f"Model: {MODEL}  |  Ready.")

Model: llama3.2:1b  |  Ready.


## 2. Core Utilities

In [4]:
def llm(prompt: str, system: str = "", stream: bool = False) -> str:
    """Call Ollama synchronously."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    if stream:
        response = ""
        for chunk in ollama.chat(model=MODEL, messages=messages, stream=True):
            piece = chunk["message"]["content"]
            print(piece, end="", flush=True)
            response += piece
        print()
        return response
    else:
        result = ollama.chat(model=MODEL, messages=messages)
        return result["message"]["content"]


def search_web(query: str, max_results: int = MAX_SEARCH_RESULTS) -> list[dict]:
    """Search DuckDuckGo and return list of {title, url, snippet}."""
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({
                    "title": r.get("title", ""),
                    "url": r.get("href", ""),
                    "snippet": r.get("body", ""),
                })
    except Exception as e:
        print(f"  [Search error: {e}]")
    return results


def fetch_page(url: str) -> str:
    """Fetch and extract clean text from a URL."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; DeepSearch/1.0)"}
    try:
        resp = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        # Remove boilerplate tags
        for tag in soup(["script", "style", "nav", "footer", "header",
                         "aside", "form", "iframe", "noscript"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
        # Collapse whitespace
        text = re.sub(r"\s+", " ", text)
        return text
    except Exception as e:
        return ""


def chunk_text(text: str, chunk_size: int = MAX_CHUNK_CHARS) -> list[str]:
    """Split text into overlapping chunks."""
    chunks = []
    step = int(chunk_size * 0.8)
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if len(chunk) > 200:  # skip tiny chunks
            chunks.append(chunk)
    return chunks


print("Utilities defined.")

Utilities defined.


## 3. DeepSearch Pipeline

In [5]:
def generate_sub_queries(topic: str) -> list[str]:
    """Ask the LLM to generate targeted search queries for the topic."""
    print("[1/4] Generating sub-queries...")
    prompt = f"""Generate {MAX_SUB_QUERIES} specific, diverse search queries to thoroughly research:
"{topic}"

Return ONLY the queries, one per line, no numbering, no explanation."""
    raw = llm(prompt)
    queries = [q.strip() for q in raw.strip().splitlines() if q.strip()]
    queries = queries[:MAX_SUB_QUERIES]
    # Always include the original topic
    if topic not in queries:
        queries.insert(0, topic)
    print(f"  Queries: {queries}")
    return queries


def gather_sources(queries: list[str]) -> list[dict]:
    """Search and deduplicate URLs."""
    print("[2/4] Searching the web...")
    seen_urls = set()
    all_results = []
    for q in queries:
        results = search_web(q)
        for r in results:
            if r["url"] and r["url"] not in seen_urls:
                seen_urls.add(r["url"])
                all_results.append(r)
        time.sleep(0.5)  # polite delay
    print(f"  Found {len(all_results)} unique sources.")
    return all_results


def scrape_and_rank(sources: list[dict], topic: str) -> list[str]:
    """Fetch pages, chunk, and score relevance with LLM."""
    print("[3/4] Scraping pages & ranking chunks...")
    scored_chunks = []
    
    for i, src in enumerate(sources):
        url = src["url"]
        print(f"  [{i+1}/{len(sources)}] {url[:80]}")
        
        # Use snippet as fallback if scraping fails
        page_text = fetch_page(url)
        if not page_text:
            page_text = src.get("snippet", "")
        
        if not page_text:
            continue
        
        chunks = chunk_text(page_text)
        
        # Score each chunk for relevance
        for chunk in chunks[:4]:  # limit chunks per page to save time
            score_prompt = f"""Rate how relevant this text is to the topic "{topic}" on a scale 0-10.
Return ONLY a number.

Text: {chunk[:500]}"""
            try:
                score_str = llm(score_prompt).strip()
                score = float(re.search(r"\d+\.?\d*", score_str).group())
            except:
                score = 5.0
            
            scored_chunks.append((score, chunk, src["title"], url))
    
    # Sort by relevance descending
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    top = scored_chunks[:TOP_CHUNKS]
    
    print(f"  Selected {len(top)} top chunks (scores: {[round(s,1) for s,*_ in top]})")
    return top


def synthesize_report(topic: str, top_chunks: list) -> str:
    """Generate the final deep research report."""
    print("[4/4] Synthesizing report (streaming)...\n")
    
    context_parts = []
    for score, chunk, title, url in top_chunks:
        context_parts.append(f"--- Source: {title} ({url}) ---\n{chunk}")
    context = "\n\n".join(context_parts)
    
    system = """You are a thorough research assistant. Write a comprehensive, well-structured report 
based on the provided sources. Use markdown with headers, bullet points, and clear sections. 
Cite sources by title when referencing specific information. Be factual and detailed."""
    
    prompt = f"""Based on the following research sources, write a comprehensive deep-dive report on:
**{topic}**

SOURCES:
{context}

Write the report now:"""
    
    return llm(prompt, system=system, stream=True)


def deep_search(topic: str):
    """Main DeepSearch entry point."""
    print(f"\n{'='*60}")
    print(f"DeepSearch: {topic}")
    print(f"{'='*60}\n")
    
    queries = generate_sub_queries(topic)
    sources = gather_sources(queries)
    top_chunks = scrape_and_rank(sources, topic)
    report = synthesize_report(topic, top_chunks)
    
    print(f"\n{'='*60}")
    print("Done.")
    return report


print("Pipeline defined.")

Pipeline defined.


## 4. Run DeepSearch

Change the `TOPIC` variable and run the cell.

In [6]:
# ---- SET YOUR TOPIC HERE ----
TOPIC = "quantum computing recent breakthroughs 2024"
# -----------------------------

report = deep_search(TOPIC)


DeepSearch: quantum computing recent breakthroughs 2024

[1/4] Generating sub-queries...
  Queries: ['quantum computing recent breakthroughs 2024', 'Here are three specific, diverse search queries:', '1. quantum computing latest news and breakthroughs 2024', '2. new quantum computing innovations and developments 2024']
[2/4] Searching the web...
  Found 24 unique sources.
[3/4] Scraping pages & ranking chunks...
  [1/24] https://fucoder.com/latest-breakthroughs-in-quantum-computing-2024/
  [2/24] https://willametteweekly.news/quantum-computing-breakthroughs-of-june-2024-a-new
  [3/24] https://techtobard.com/recent-advances-in-quantum-computing/
  [4/24] https://gimkit.blog/latest-breakthroughs-in-quantum-computing-2024/
  [5/24] https://techworlzupdates.com/how-quantum-computing-hardware-is-redefining-techno
  [6/24] https://www.weblogicnet.com/how-quantum-computing-hardware-is-redefining-technol
  [7/24] https://n8nworkflows.xyz/workflows/perplexity-style-iterative-research-with-gemi

In [7]:
# Render the final report as formatted Markdown
display(Markdown(report))

**The Road to Shor Era: A Critical Assessment of Published Roadmaps**

As the year 2024 comes to a close, it's essential to evaluate published roadmaps for quantum computing and assess their potential impact on the industry. In this report, we'll analyze three prominent quantum computing roadmaps, focusing on technical details, performance targets, and accountability frameworks.

**Roadmap 1: QuEra**

QuEra's roadmap is notable for incorporating a new error correction method using transversal gates. This innovation aims to reduce error rates by leveraging physical qubits, which are more robust than quantum bits (qubits). Key features include:

* Development of a reliable logical qubit using eight physical qubits
* Achievement of 10 logical qubits by the close of 2024

While this roadmap demonstrates significant technical progress, we're skeptical about its potential impact. The achievement of only 10 logical qubits may not be sufficient to overcome current computational challenges.

**Roadmap 2: Google's Quantum AI Lab**

Google's roadmap focuses on advancing artificial intelligence (AI) and quantum computing in tandem. Key features include:

* Establishment of a new quantum AI lab with 100+ researchers
* Development of practical applications for quantum AI, such as machine learning and optimization
* Potential collaboration with industry partners to leverage AI-driven quantum solutions

While this roadmap showcases Google's commitment to interdisciplinary research, we question its potential success. Integrating AI and quantum computing may require significant breakthroughs in both fields.

**Roadmap 3: IBM Quantum Experience**

IBM's roadmap emphasizes the development of practical applications for quantum computing in industries like finance, healthcare, and materials science. Key features include:

* Establishment of a new quantum computing platform with 256 qubits
* Development of cloud-based quantum software frameworks and tools
* Potential collaboration with industry partners to leverage quantum computing solutions

We're impressed by IBM's focus on practical applications and cloud-based services. However, the scope of this roadmap may be limited compared to other contenders.

**Accountability Frameworks**

The introduction of published roadmaps creates a new accountability framework for the quantum computing industry:

* Quantifiable progress: Specific numbers for error rates, qubit counts, and operational metrics
* Technical transparency: Architectural details and specific implementation strategies
* Time-bound commitments: Clear yearly milestones through 2025-2030

These frameworks ensure that success will no longer be judged on incremental progress or isolated demonstrations. Instead, the industry can rely on publicly available roadmaps to evaluate their technical feasibility.

**Conclusion**

The year 2024 has brought significant advancements in quantum computing, including error correction methods and practical applications for artificial intelligence. However, we caution against overly optimistic expectations based on these early milestones. The next five years (2024-2029) will be crucial for the quantum computing industry to deliver on its promises.

As the industry enters this new era of delivery, it's essential to evaluate published roadmaps through a critical lens. By assessing technical details, performance targets, and accountability frameworks, we can better understand the potential impact of these initiatives on the quantum computing landscape.

**Recommendations**

To accelerate progress in the quantum computing industry:

1. **Foster collaboration**: Encourage cross-industry partnerships to leverage AI-driven quantum solutions.
2. **Invest in fundamental research**: Continuously support research in areas like error correction, scalable qubits, and algorithmic innovation.
3. **Develop practical applications**: Focus on developing practical applications for quantum computing in industries like finance, healthcare, and materials science.

By prioritizing these recommendations, the industry can unlock the full potential of quantum computing and create a new era of technological advancements.

## 5.5  📚 Personalised Learning Roadmap

After deep-search, the LLM extracts a structured roadmap and stores it in a **Neo4j graph database**.  
A graph-based recommendation algorithm then ranks the next best modules for the learner.

In [8]:
!pip install neo4j -q

In [10]:
import json, re
from neo4j import GraphDatabase

NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "deepsearch"

def get_neo4j_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# ── test connection ──
try:
    _drv = get_neo4j_driver()
    _drv.verify_connectivity()
    print("✅ Neo4j connected")
    _drv.close()
    neo4j_ok = True
except Exception as _e:
    print(f"❌ Neo4j not available: {_e}")
    print("   Start with:  docker run -d --name deepsearch-neo4j \\")
    print("     -p 7474:7474 -p 7687:7687 \\")
    print("     -e NEO4J_AUTH=neo4j/deepsearch neo4j:5")
    neo4j_ok = False

✅ Neo4j connected


In [11]:
def extract_learning_roadmap(topic: str, report: str) -> dict:
    """Ask the LLM to parse the deepsearch report into a structured roadmap JSON."""
    prompt = f"""You are a curriculum designer.
Based on the research report about "{topic}", create a personalised learning roadmap.

Return ONLY valid JSON — no markdown fences, no extra text:
{{
  "topic": "{topic}",
  "level": "beginner",
  "level_emoji": "🟢",
  "gaps": ["gap 1", "gap 2"],
  "modules": [
    {{
      "id": 1,
      "title": "Module title",
      "objective": "Clear learning objective",
      "concepts": ["c1", "c2"],
      "duration_minutes": 60,
      "prerequisites": []
    }}
  ]
}}

Rules:
• level must be "beginner", "intermediate", or "advanced"
• level_emoji: 🟢 beginner · 🟡 intermediate · 🔴 advanced
• 4–7 modules, ordered foundational → advanced
• prerequisites: list of module ids that must be done first
• duration_minutes: 30 / 45 / 60 / 90 / 120
• gaps: 2–4 key knowledge areas this roadmap addresses

Report (excerpt):
{report[:3500]}"""

    raw = llm(prompt)
    match = re.search(r'\{{[\s\S]*\}}', raw)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    # ── graceful fallback ──
    return {
        "topic": topic,
        "level": "beginner",
        "level_emoji": "🟢",
        "gaps": [f"fundamentals of {topic}"],
        "modules": [{
            "id": 1,
            "title": f"{topic}: Introduction",
            "objective": f"Understand the core concepts of {topic}",
            "concepts": ["overview", "fundamentals"],
            "duration_minutes": 60,
            "prerequisites": []
        }]
    }

In [12]:
# ── Graph schema ─────────────────────────────────────────────────────────────
def init_graph_schema(driver):
    with driver.session() as s:
        s.run("CREATE CONSTRAINT topic_unique   IF NOT EXISTS FOR (t:Topic)   REQUIRE t.name IS UNIQUE")
        s.run("CREATE CONSTRAINT module_unique  IF NOT EXISTS FOR (m:Module)  REQUIRE m.uid  IS UNIQUE")
        s.run("CREATE CONSTRAINT concept_unique IF NOT EXISTS FOR (c:Concept) REQUIRE c.name IS UNIQUE")
        s.run("CREATE CONSTRAINT level_unique   IF NOT EXISTS FOR (l:Level)   REQUIRE l.name IS UNIQUE")
    print("✅ Graph schema ready")


# ── Persist roadmap as graph ──────────────────────────────────────────────────
def save_roadmap_to_graph(driver, roadmap: dict):
    """
    Nodes   : Topic · Level · Module · Concept
    Edges   : TARGET_LEVEL · HAS_MODULE · PREREQUISITE_FOR · TEACHES
    """
    topic = roadmap["topic"]
    level = roadmap["level"]

    with driver.session() as s:
        # Topic & Level nodes
        s.run("MERGE (t:Topic {name:$n}) SET t.level=$lv, t.emoji=$em",
              n=topic, lv=level, em=roadmap["level_emoji"])
        s.run("MERGE (l:Level {name:$n})", n=level)
        s.run("MATCH (t:Topic{name:$t}),(l:Level{name:$l}) MERGE (t)-[:TARGET_LEVEL]->(l)",
              t=topic, l=level)

        for mod in roadmap["modules"]:
            uid = f"{topic}::{mod['id']}"
            # Module node
            s.run("""
                MERGE (m:Module {uid:$uid})
                SET m.title=$title, m.objective=$obj,
                    m.duration_minutes=$dur, m.order=$n, m.topic=$topic
            """, uid=uid, title=mod["title"], obj=mod["objective"],
                  dur=mod["duration_minutes"], n=mod["id"], topic=topic)

            # Topic → Module
            s.run("MATCH (t:Topic{name:$t}),(m:Module{uid:$u}) MERGE (t)-[:HAS_MODULE]->(m)",
                  t=topic, u=uid)

            # Module → Concepts
            for cname in mod.get("concepts", []):
                s.run("MERGE (c:Concept {name:$n})", n=cname)
                s.run("MATCH (m:Module{uid:$u}),(c:Concept{name:$n}) MERGE (m)-[:TEACHES]->(c)",
                      u=uid, n=cname)

            # Prerequisite edges
            for pid in mod.get("prerequisites", []):
                pre_uid = f"{topic}::{pid}"
                s.run("MATCH (pre:Module{uid:$p}),(m:Module{uid:$u}) MERGE (pre)-[:PREREQUISITE_FOR]->(m)",
                      p=pre_uid, u=uid)

    n = len(roadmap["modules"])
    print(f"✅ Saved to graph — topic='{topic}', {n} modules, level={level}")


# ── Recommendation algorithm ──────────────────────────────────────────────────
def recommend_modules(driver, topic: str, completed_ids: list = None) -> list:
    """
    Graph-traversal recommender:
    1. Find modules whose prerequisites are all satisfied (completed).
    2. Rank by (concepts_taught DESC, module_order ASC) — prefer breadth-first coverage.
    Returns a list of dicts ready for display.
    """
    completed_ids = completed_ids or []
    completed_uids = [f"{topic}::{i}" for i in completed_ids]

    with driver.session() as s:
        result = s.run("""
            MATCH (t:Topic {name:$topic})-[:HAS_MODULE]->(m:Module)
            WHERE NOT m.uid IN $done
              AND NOT EXISTS {
                  MATCH (pre:Module)-[:PREREQUISITE_FOR]->(m)
                  WHERE NOT pre.uid IN $done
              }
            OPTIONAL MATCH (m)-[:TEACHES]->(c:Concept)
            WITH m, count(c) AS concepts_taught
            RETURN m.uid          AS uid,
                   m.title        AS title,
                   m.objective    AS objective,
                   m.duration_minutes AS duration,
                   m.order        AS module_order,
                   concepts_taught
            ORDER BY concepts_taught DESC, module_order ASC
        """, topic=topic, done=completed_uids)
        return [dict(r) for r in result]

In [13]:
def display_learning_roadmap(roadmap: dict, recommendations: list = None):
    """Render the personalised roadmap as a rich markdown table."""
    topic   = roadmap["topic"]
    level   = roadmap["level"].capitalize()
    emoji   = roadmap["level_emoji"]
    gaps    = roadmap["gaps"]
    modules = roadmap["modules"]

    total_min = sum(m["duration_minutes"] for m in modules)
    h, m_rem  = divmod(total_min, 60)
    total_str = (f"{h}h {m_rem:02d}min" if h else f"{m_rem}min")

    lines = [
        f"## 📚 Your Personalised Learning Roadmap",
        f"**Topic:** {topic} · **Level:** {emoji} {level}",
        "",
        "**Knowledge gaps addressed:**",
        *[f"- {g}" for g in gaps],
        "",
        "| # | Module | Objective | Concepts | Duration |",
        "|---|--------|-----------|----------|----------|",
    ]

    for mod in modules:
        concepts = ", ".join(mod["concepts"])
        dur_str  = f"⏱ {mod['duration_minutes']} minutes"
        prereq_note = ""
        if mod.get("prerequisites"):
            prereq_note = f" *(req: {', '.join(map(str, mod['prerequisites']))})*"
        lines.append(
            f"| {mod['id']} | **{mod['title']}** | {mod['objective']}{prereq_note} | {concepts} | {dur_str} |"
        )

    lines += ["", f"**{len(modules)} modules · estimated total: {total_str}**"]

    if recommendations:
        lines += [
            "",
            "---",
            "### 🎯 Recommended Next Modules",
            "",
            "| Priority | Module | Concepts Covered | Duration |",
            "|----------|--------|-----------------|----------|",
        ]
        for rank, r in enumerate(recommendations[:3], 1):
            lines.append(
                f"| {rank} | **{r['title']}** | {r['concepts_taught']} concepts | ⏱ {r['duration']} min |"
            )

    display(Markdown("\n".join(lines)))

In [14]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
LR_COMPLETED = []   # e.g. [1, 2]  — list module IDs already completed

# ── Extract roadmap from the deepsearch report ────────────────────────────────
print("🔍 Extracting learning roadmap from report…")
roadmap = extract_learning_roadmap(TOPIC, report)

# ── Store in Neo4j & get recommendations ─────────────────────────────────────
recommendations = []
if neo4j_ok:
    drv = get_neo4j_driver()
    init_graph_schema(drv)
    save_roadmap_to_graph(drv, roadmap)
    recommendations = recommend_modules(drv, TOPIC, LR_COMPLETED)
    drv.close()
else:
    print("⚠️  Neo4j offline — recommendations skipped (roadmap still shown)")

# ── Display ───────────────────────────────────────────────────────────────────
display_learning_roadmap(roadmap, recommendations)

🔍 Extracting learning roadmap from report…
✅ Graph schema ready
✅ Saved to graph — topic='quantum computing recent breakthroughs 2024', 1 modules, level=beginner


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `PREREQUISITE_FOR` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=5, column=40, offset=176>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 176, 'line': 5, 'column': 40}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (t:Topic {name:$topic})-[:HAS_MODULE]->(m:Module)\n            WHERE NOT m.uid IN $done\n              AND NOT EXISTS {\n                  MATCH (pre:Module)-[:PREREQUISITE_FOR]->(m)\n                  WHERE NOT pre.uid IN $done\n              }\n            OPTIONAL MATCH (m)-[:TEACHES]->

## 📚 Your Personalised Learning Roadmap
**Topic:** quantum computing recent breakthroughs 2024 · **Level:** 🟢 Beginner

**Knowledge gaps addressed:**
- fundamentals of quantum computing recent breakthroughs 2024

| # | Module | Objective | Concepts | Duration |
|---|--------|-----------|----------|----------|
| 1 | **quantum computing recent breakthroughs 2024: Introduction** | Understand the core concepts of quantum computing recent breakthroughs 2024 | overview, fundamentals | ⏱ 60 minutes |

**1 modules · estimated total: 1h 00min**

---
### 🎯 Recommended Next Modules

| Priority | Module | Concepts Covered | Duration |
|----------|--------|-----------------|----------|
| 1 | **quantum computing recent breakthroughs 2024: Introduction** | 2 concepts | ⏱ 60 min |

## 5. Quick Search (no scraping — fast mode)

In [15]:
def quick_search(topic: str):
    """Faster version: use DuckDuckGo snippets only, no full page scraping."""
    print(f"Quick search: {topic}\n")
    
    results = search_web(topic, max_results=10)
    snippets = "\n\n".join(
        f"**{r['title']}** ({r['url']})\n{r['snippet']}"
        for r in results if r['snippet']
    )
    
    system = "You are a research assistant. Summarize and synthesize the search results into a clear, structured answer."
    prompt = f"Topic: {topic}\n\nSearch results:\n{snippets}\n\nWrite a comprehensive summary:"
    
    return llm(prompt, system=system, stream=True)


# ---- SET YOUR TOPIC HERE ----
QUICK_TOPIC = "latest AI models 2025"
# -----------------------------

quick_report = quick_search(QUICK_TOPIC)
display(Markdown(quick_report))

Quick search: latest AI models 2025

Based on the search results, it appears that several AI models are expected to be released or become available in 2025, with some already being developed and others still in research stages. Here's a comprehensive summary of the latest AI models predicted for 2025:

**Top AI Models 2025:**

1. **DeepSeek V3**: An open-source AI model with 671B total parameters, offering a high level of performance.
2. **Google's Gemini 2.0**: A more advanced version of Google's AI model, incorporating additional functionalities and capabilities.
3. **Anthropic Claude**: A highly capable AI model developed by Anthropic, designed to excel in various tasks.

**Breakdown of Most Powerful AI Models Launched in 2025:**

1. **OpenAI's Orion**: An internal model at OpenAI, expected to perform well in handling language understanding and generation tasks.
2. **ChatGPT**: A rival AI model from Microsoft, aiming to provide more human-like conversation capabilities.
3. **CLAude*

Based on the search results, it appears that several AI models are expected to be released or become available in 2025, with some already being developed and others still in research stages. Here's a comprehensive summary of the latest AI models predicted for 2025:

**Top AI Models 2025:**

1. **DeepSeek V3**: An open-source AI model with 671B total parameters, offering a high level of performance.
2. **Google's Gemini 2.0**: A more advanced version of Google's AI model, incorporating additional functionalities and capabilities.
3. **Anthropic Claude**: A highly capable AI model developed by Anthropic, designed to excel in various tasks.

**Breakdown of Most Powerful AI Models Launched in 2025:**

1. **OpenAI's Orion**: An internal model at OpenAI, expected to perform well in handling language understanding and generation tasks.
2. **ChatGPT**: A rival AI model from Microsoft, aiming to provide more human-like conversation capabilities.
3. **CLAude**: Another top-rated AI model, developed by Anthropic, designed for various applications such as conversational AI and content creation.

**Ranking of the Best AI Models in 2025:**

1. **Gemini 2.0**: Google's latest AI model, with additional functionalities and capabilities.
2. **DeepSeek V3**: An open-source AI model offering high performance.
3. **Anthropic Claude**: A highly capable AI model for various applications.

**Key Takeaways:**

* Several AI models are expected to be released or become available in 2025, including DeepSeek V3, Google's Gemini 2.0, and Anthropic Claude.
* These models will have varying levels of performance, with some being more advanced than others.
* OpenAI's Orion is also predicted to perform well in handling language understanding and generation tasks.
* AI companies are struggling to improve their latest models, highlighting the challenges and limitations of current AI technology.

**Conclusion:**

The search results indicate that several top-rated AI models will be released or become available in 2025. These models have varying levels of performance and capabilities, with some being more advanced than others. As AI technology continues to evolve, it is likely that we will see continued improvements and advancements in this field.

## 6. YouTube Search — Video IDs & Metadata

In [1]:
print("YouTube section ready. Use search_youtube(query) below.")

YouTube section ready. Use search_youtube(query) below.


In [6]:
def search_youtube(query, max_results=YT_MAX_RESULTS):
    """Search YouTube without downloading. Returns video id, title, channel, duration, views, url, thumbnail."""
    ydl_opts = {"quiet": True, "no_warnings": True, "extract_flat": True}
    videos = []
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(f"ytsearch{max_results}:{query}", download=False)
        for entry in info.get("entries", []):
            d = int(entry.get("duration") or 0)
            vid_id = entry.get("id", "")
            videos.append({
                "id":       vid_id,
                "title":    entry.get("title", ""),
                "channel":  entry.get("channel") or entry.get("uploader", ""),
                "duration": f"{d // 60}:{d % 60:02d}",
                "views":    entry.get("view_count"),
                "desc":     (entry.get("description") or "")[:250],
                "url":      "https://www.youtube.com/watch?v=" + vid_id,
                "thumb":    "https://img.youtube.com/vi/" + vid_id + "/hqdefault.jpg",
            })
    return videos


print("search_youtube() ready.")

search_youtube() ready.


In [7]:
def display_youtube_results(videos):
    """Render results as a Markdown table + plain ID list."""
    if not videos:
        print("No results.")
        return
    rows = [
        "| # | Title | Channel | Duration | Views | ID |",
        "|---|-------|---------|----------|-------|----|",
    ]
    for i, v in enumerate(videos, 1):
        views = f"{v['views']:,}" if v["views"] else "N/A"
        title = f"[{v['title'][:55]}]({v['url']})"
        rows.append(f"| {i} | {title} | {v['channel']} | {v['duration']} | {views} | `{v['id']}` |")
    display(Markdown("\n".join(rows)))
    print("\nVideo IDs:")
    for v in videos:
        print(f"  {v['id']}   {v['url']}")
        if v["desc"]:
            print(f"  {v['desc'][:110]}")
        print()


print("display_youtube_results() ready.")

display_youtube_results() ready.


In [12]:
# ---- SET YOUR YOUTUBE QUERY HERE ----
YT_QUERY = "quantum computing explained 2024"
# --------------------------------------

yt_videos = search_youtube(YT_QUERY)
display_youtube_results(yt_videos)

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Quantum Computers Explained: How Quantum Computing Work](https://www.youtube.com/watch?v=B3U1NDUiwSA) | Science ABC | 5:41 | 733,457 | `B3U1NDUiwSA` |
| 2 | [Quantum Computing Explained in 2 Minutes 2024](https://www.youtube.com/watch?v=bEYK8gxuFqs) | @TechTomenia | 1:19 | 4 | `bEYK8gxuFqs` |
| 3 | [Quantum Computing Explained: 20 Ways It Will Affect EVE](https://www.youtube.com/watch?v=79kNOf749MA) | Future Business Tech | 86:29 | 346,505 | `79kNOf749MA` |
| 4 | [Brian Cox on quantum computing and black hole physics](https://www.youtube.com/watch?v=laGXRs9Ce70) | Big Think | 6:43 | 1,088,122 | `laGXRs9Ce70` |
| 5 | [Quantum Computing 2025 Update](https://www.youtube.com/watch?v=cDt7o-OmBWI) | ExplainingComputers | 17:05 | 61,948 | `cDt7o-OmBWI` |
| 6 | [How Physicists Proved Everything is Quantum - Nobel Phy](https://www.youtube.com/watch?v=_mVBbdbqHmw) | Dr Ben Miles | 7:18 | 1,295,422 | `_mVBbdbqHmw` |


Video IDs:
  B3U1NDUiwSA   https://www.youtube.com/watch?v=B3U1NDUiwSA
  Quantum computers use the principles of quantum mechanics to process information in ways that classical comput

  bEYK8gxuFqs   https://www.youtube.com/watch?v=bEYK8gxuFqs
  Quantum computing is a revolutionary field that leverages the principles of quantum mechanics to process infor

  79kNOf749MA   https://www.youtube.com/watch?v=79kNOf749MA
  Boost your knowledge in quantum computing and other emerging technologies with Brilliant's engaging courses. E

  laGXRs9Ce70   https://www.youtube.com/watch?v=laGXRs9Ce70
  Become a Big Think member to unlock expert classes, premium print issues, exclusive events and more: ...

  cDt7o-OmBWI   https://www.youtube.com/watch?v=cDt7o-OmBWI
  Quantum computing developments over the past 12 months, including Google Willow, IBM Majorana 1, and innovatio

  _mVBbdbqHmw   https://www.youtube.com/watch?v=_mVBbdbqHmw
  The Nobel Prize in Physics 2025 was awarded jointly to John Cl

## 7. Combined: Web Report + YouTube Videos

In [8]:
def full_research(topic):
    """Quick web summary + YouTube videos for any topic."""
    print(f"\n{'='*60}\nWEB: {topic}\n{'='*60}")
    report = quick_search(topic)
    print(f"\n{'='*60}\nYOUTUBE: {topic}\n{'='*60}")
    videos = search_youtube(topic)
    display_youtube_results(videos)
    return report, videos


# ---- SET TOPIC ----
COMBINED_TOPIC = "large language models 2025"
# -------------------

web_report, videos = full_research(COMBINED_TOPIC)
display(Markdown(web_report))


WEB: large language models 2025
Quick search: large language models 2025

**Comprehensive Summary of Large Language Models 2025**

Large Language Models (LLMs) have emerged as a significant breakthrough in the field of artificial intelligence, with advancements in transformer-based models since their inception. These models have revolutionized various applications, including natural language processing, text generation, and chatbots.

**Definition and Overview**

A Large Language Model (LLM) is a deep learning model designed to process, understand, and generate human-like text. Developed on immense amounts of data, LLMs can learn patterns, grammar, and context from text and perform tasks such as answering questions, writing content, translating languages, and more.

**Key Features and Applications**

1. **Deep Learning**: LLMs are built on deep neural networks, enabling them to process vast amounts of data and understand complex relationships between words.
2. **Contextual Understandi

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [How to Choose Large Language Models: A Developer’s Guid](https://www.youtube.com/watch?v=pYax2rupKEY) | IBM Technology | 6:57 | 94,850 | `pYax2rupKEY` |
| 2 | [Large Language Models explained briefly](https://www.youtube.com/watch?v=LPZh9BOjkQs) | 3Blue1Brown | 7:58 | 5,535,911 | `LPZh9BOjkQs` |
| 3 | [Inside GPT – Large Language Models Demystified - Alan S](https://www.youtube.com/watch?v=f6dFD1Hzv20) | NDC Conferences | 61:28 | 2,339 | `f6dFD1Hzv20` |
| 4 | [How Large Language Models Work](https://www.youtube.com/watch?v=5sLYAQS9sWQ) | IBM Technology | 5:34 | 1,399,009 | `5sLYAQS9sWQ` |
| 5 | [Christopher Manning: Large Language Models in 2025 – Ho](https://www.youtube.com/watch?v=5Aer7MUSuSU) | Stanford HAI | 39:23 | 18,417 | `5Aer7MUSuSU` |
| 6 | [Stanford CS229 I Machine Learning I Building Large Lang](https://www.youtube.com/watch?v=9vM4p9NN0Ts) | Stanford Online | 104:31 | 1,827,188 | `9vM4p9NN0Ts` |


Video IDs:
  pYax2rupKEY   https://www.youtube.com/watch?v=pYax2rupKEY
  Ready to become a certified watsonx AI Assistant Engineer? Register now and use code IBMTechYT20 for 20% off o

  LPZh9BOjkQs   https://www.youtube.com/watch?v=LPZh9BOjkQs
  A light intro to LLMs, chatbots, pretraining, and transformers. Dig deeper here: ...

  f6dFD1Hzv20   https://www.youtube.com/watch?v=f6dFD1Hzv20
  This talk was recorded at NDC AI in Oslo, Norway. #ndcai #ndcconferences #developer #softwaredeveloper Attend 

  5sLYAQS9sWQ   https://www.youtube.com/watch?v=5sLYAQS9sWQ
  Learn in-demand Machine Learning skills now → https://ibm.biz/BdK65D Learn about watsonx → https://ibm.biz/Bdv

  5Aer7MUSuSU   https://www.youtube.com/watch?v=5Aer7MUSuSU
  The Stanford Open Virtual Assistant Lab, with sponsorship from the Alfred P. Sloan Foundation and the Stanford

  9vM4p9NN0Ts   https://www.youtube.com/watch?v=9vM4p9NN0Ts
  For more information about Stanford's Artificial Intelligence programs visit: http

**Comprehensive Summary of Large Language Models 2025**

Large Language Models (LLMs) have emerged as a significant breakthrough in the field of artificial intelligence, with advancements in transformer-based models since their inception. These models have revolutionized various applications, including natural language processing, text generation, and chatbots.

**Definition and Overview**

A Large Language Model (LLM) is a deep learning model designed to process, understand, and generate human-like text. Developed on immense amounts of data, LLMs can learn patterns, grammar, and context from text and perform tasks such as answering questions, writing content, translating languages, and more.

**Key Features and Applications**

1. **Deep Learning**: LLMs are built on deep neural networks, enabling them to process vast amounts of data and understand complex relationships between words.
2. **Contextual Understanding**: They can comprehend context and nuances in language, allowing for accurate and informative responses.
3. **Text Generation**: LLMs can generate coherent text, including articles, stories, and conversations.
4. **Chatbots and Virtual Assistants**: They are used to power chatbots and virtual assistants, enabling users to interact with AI-powered systems.

**Advantages and Limitations**

Advantages:

* Improved language understanding and generation
* Increased productivity and efficiency in various applications
* Enhanced creativity and innovation

Limitations:

* Lack of human-like empathy and emotional intelligence
* Dependence on high-quality training data
* Potential for bias in the training data, leading to unfair or discriminatory outcomes

**Recent Advancements**

1. **Aligning Large Language Models with Human Opinions**: Researchers have developed techniques to align LLMs with human opinions through persona selection and value-belief reasoning.
2. **Limitations of LLMs**: Studies have shown that LLMs are not yet good enough to imitate the way people speak, making them useful for specific tasks but not suitable for tasks requiring human-like understanding.

**Future Directions**

1. **Hybrid Models**: Combining LLMs with other AI models, such as cognitive architectures or rule-based systems, to improve overall performance.
2. **Explainability and Transparency**: Developing techniques to explain the reasoning behind LLM output, enabling better trustworthiness and accountability.
3. **Human-AI Collaboration**: Designing systems that facilitate human-AI collaboration, enhancing productivity and efficiency in various applications.

**Conclusion**

Large Language Models have transformed the field of artificial intelligence, offering numerous benefits and promising future developments. While they face limitations, ongoing research aims to address these challenges and improve their performance. As LLMs continue to evolve, it is essential to consider their strengths and weaknesses, as well as the social implications of AI adoption.

In [ ]:
## 8. PostgreSQL Storage (Docker)\n\nAll searches, results and reports are auto-saved.\n\n**Start the container once:**\n```bash\ndocker run -d --name deepsearch-pg \\\n  -e POSTGRES_USER=deepsearch \\\n  -e POSTGRES_PASSWORD=deepsearch \\\n  -e POSTGRES_DB=deepsearch \\\n  -p 5432:5432 postgres:16-alpine\n```

In [29]:
!pip install psycopg2-binary SQLAlchemy -q

In [23]:
!docker start deepsearch-pg


deepsearch-pg


In [9]:
import psycopg2
from psycopg2.extras import RealDictCursor

DB = dict(host="localhost", port=5432, dbname="deepsearch", user="deepsearch", password="deepsearch")

def get_conn():
    return psycopg2.connect(**DB)

def init_db():
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                CREATE TABLE IF NOT EXISTS searches (
                    id          SERIAL PRIMARY KEY,
                    topic       TEXT NOT NULL,
                    search_type TEXT NOT NULL,
                    created_at  TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS web_results (
                    id        SERIAL PRIMARY KEY,
                    search_id INT REFERENCES searches(id) ON DELETE CASCADE,
                    title     TEXT,
                    url       TEXT,
                    snippet   TEXT,
                    created_at TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS youtube_results (
                    id          SERIAL PRIMARY KEY,
                    search_id   INT REFERENCES searches(id) ON DELETE CASCADE,
                    video_id    TEXT,
                    title       TEXT,
                    channel     TEXT,
                    duration    TEXT,
                    views       BIGINT,
                    description TEXT,
                    url         TEXT,
                    thumbnail   TEXT,
                    created_at  TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS reports (
                    id        SERIAL PRIMARY KEY,
                    search_id INT REFERENCES searches(id) ON DELETE CASCADE,
                    content   TEXT,
                    created_at TIMESTAMPTZ DEFAULT NOW()
                );
            """)
        conn.commit()
    print("DB ready — tables: searches, web_results, youtube_results, reports")

init_db()

DB ready — tables: searches, web_results, youtube_results, reports


In [10]:
def _new_search(topic, kind):
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("INSERT INTO searches (topic, search_type) VALUES (%s,%s) RETURNING id", (topic, kind))
            sid = cur.fetchone()[0]
        conn.commit()
    return sid

def _save_web(sid, results):
    if not results: return
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.executemany(
                "INSERT INTO web_results (search_id,title,url,snippet) VALUES (%s,%s,%s,%s)",
                [(sid, r["title"], r["url"], r["snippet"]) for r in results]
            )
        conn.commit()

def _save_yt(sid, videos):
    if not videos: return
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.executemany(
                """INSERT INTO youtube_results
                   (search_id,video_id,title,channel,duration,views,description,url,thumbnail)
                   VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
                [(sid, v["id"], v["title"], v["channel"], v["duration"],
                  v["views"], v["desc"], v["url"], v["thumb"]) for v in videos]
            )
        conn.commit()

def _save_report(sid, text):
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("INSERT INTO reports (search_id,content) VALUES (%s,%s)", (sid, text))
        conn.commit()

print("DB helpers ready.")

DB helpers ready.


In [11]:
def db_quick_search(topic):
    """Quick web search + save to DB."""
    sid = _new_search(topic, "quick")
    results = search_web(topic, max_results=10)
    _save_web(sid, results)
    snippets = "\n\n".join(f"**{r['title']}**\n{r['snippet']}" for r in results if r["snippet"])
    report = llm(f"Topic: {topic}\n\nResults:\n{snippets}\n\nSummary:",
                 system="Summarize into a clear structured answer.", stream=True)
    _save_report(sid, report)
    print(f"\n[DB] search_id={sid} | {len(results)} web results + report saved")
    return sid, report

def db_youtube_search(topic):
    """YouTube search + save to DB."""
    sid = _new_search(topic, "youtube")
    videos = search_youtube(topic)
    _save_yt(sid, videos)
    print(f"[DB] search_id={sid} | {len(videos)} YouTube videos saved")
    display_youtube_results(videos)
    return sid, videos

def db_full_research(topic):
    """Web + YouTube + report — everything saved to DB."""
    sid = _new_search(topic, "combined")
    # Web
    results = search_web(topic, max_results=10)
    _save_web(sid, results)
    snippets = "\n\n".join(f"**{r['title']}**\n{r['snippet']}" for r in results if r["snippet"])
    report = llm(f"Topic: {topic}\n\nResults:\n{snippets}\n\nSummary:",
                 system="Summarize into a clear structured answer.", stream=True)
    _save_report(sid, report)
    # YouTube
    videos = search_youtube(topic)
    _save_yt(sid, videos)
    print(f"\n[DB] search_id={sid} | {len(results)} web + {len(videos)} YouTube + report saved")
    display_youtube_results(videos)
    display(Markdown(report))
    return sid

print("DB search functions ready.")

DB search functions ready.


In [22]:
# ---- RUN WITH DB STORAGE ----
DB_TOPIC = "i want to learn physics and electromagnetism"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

Here's a clear and structured answer to your question:

**Learning Physics and Electromagnetism**

To learn physics and electromagnetism, it's essential to start with the basics. Here are some steps you can follow:

1. **Understand the concepts**: Start by understanding the fundamental concepts of physics, such as motion, energy, and forces. You can use online resources like [The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism](https://www.feynmanlectures.net/) to get a grasp on these ideas.
2. **Find a suitable textbook**: Once you have a good understanding of the concepts, find a textbook that covers electromagnetism and electricity. Some recommended textbooks include [AP Physics C: Electricity and Magnetism - AP Students](https://www Courseasy.com/learn/physics/ap-physics-c-electricity-and-magnetism/) and [How to Learn Physics for Free](https://www.apphysicsc.com/physicist/learn).
3. **Watch online resources**: There are many online resources available to learn electroma

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [6 Books to Self-Teach Electromagnetic Physics](https://www.youtube.com/watch?v=S6Q2Da2Ku8E) | Ali the Dazzling | 7:23 | 67,368 | `S6Q2Da2Ku8E` |
| 2 | [Electromagnetism Explained in Simple Words](https://www.youtube.com/watch?v=nllCgjlWAF4) | Science ABC | 4:14 | 283,319 | `nllCgjlWAF4` |
| 3 | [Why Physics Is Hard](https://www.youtube.com/watch?v=8fjWafOajjc) | Professor Hafner | 2:37 | 738,903 | `8fjWafOajjc` |
| 4 | [Want to study physics? Read these 10 books](https://www.youtube.com/watch?v=p9s2fBYA4fU) | Simon Clark | 14:16 | 2,286,423 | `p9s2fBYA4fU` |
| 5 | [How To Study Hard - Richard Feynman](https://www.youtube.com/watch?v=YDV1mo7QlnA) | Arjun Kocher | 3:19 | 4,732,839 | `YDV1mo7QlnA` |
| 6 | [Physics for Absolute Beginners](https://www.youtube.com/watch?v=iE-M56CVJx4) | The Math Sorcerer | 13:06 | 386,835 | `iE-M56CVJx4` |


Video IDs:
  S6Q2Da2Ku8E   https://www.youtube.com/watch?v=S6Q2Da2Ku8E
  Electromagnetic physics is the most important discipline to understand for electrical engineering students. Sa

  nllCgjlWAF4   https://www.youtube.com/watch?v=nllCgjlWAF4
  Electromagnetism is a branch of physics that deals with the study of electromagnetic forces, including electri

  8fjWafOajjc   https://www.youtube.com/watch?v=8fjWafOajjc
  This is an intro video from my online classes.

  p9s2fBYA4fU   https://www.youtube.com/watch?v=p9s2fBYA4fU
  Books for physics students! Popular science books and textbooks to get you from high school to university. Als

  YDV1mo7QlnA   https://www.youtube.com/watch?v=YDV1mo7QlnA
  Study hard what interests you the most in the most undisciplined, irreverent and original manner possible. - R

  iE-M56CVJx4   https://www.youtube.com/watch?v=iE-M56CVJx4
  This video will show you some books you can use to help get started with physics. Do you have any other recomm



Here's a clear and structured answer to your question:

**Learning Physics and Electromagnetism**

To learn physics and electromagnetism, it's essential to start with the basics. Here are some steps you can follow:

1. **Understand the concepts**: Start by understanding the fundamental concepts of physics, such as motion, energy, and forces. You can use online resources like [The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism](https://www.feynmanlectures.net/) to get a grasp on these ideas.
2. **Find a suitable textbook**: Once you have a good understanding of the concepts, find a textbook that covers electromagnetism and electricity. Some recommended textbooks include [AP Physics C: Electricity and Magnetism - AP Students](https://www Courseasy.com/learn/physics/ap-physics-c-electricity-and-magnetism/) and [How to Learn Physics for Free](https://www.apphysicsc.com/physicist/learn).
3. **Watch online resources**: There are many online resources available to learn electromagnetism, including videos, tutorials, and interactive courses. Some recommended resources include [Magnetism and Electromagnetism | AP/Crash Course Physics 2](https://www.youtube.com/watch?v=1xY6Bf7yHmQ) and [Learn Electromagnetism - Free Interactive Course - Courseasy](https://courseasy.com/learn/electromagnetism).
4. **Join a community**: Joining a community of physics students or enthusiasts can be very helpful in learning electromagnetism. You can find online forums, social media groups, or Reddit communities dedicated to physics and electromagnetism.
5. **Practice, practice, practice**: Practice is key when it comes to learning physics and electromagnetism. Start with simple problems and gradually move on to more complex ones.

**Additional Tips**

* Make sure you have a good understanding of math concepts before diving into electromagnetism.
* Use online resources like Khan Academy or MIT OpenCourseWare for additional help.
* Don't be afraid to ask questions or seek help from your teachers or mentors.
* Stay motivated and patient - learning physics and electromagnetism takes time and effort.

By following these steps, you can learn physics and electromagnetism in a structured and effective way.


All data stored under search_id = 5


In [23]:
# ---- RUN WITH DB STORAGE ----
DB_TOPIC = "i want to learn physics and electromagnetism give me resources and videos"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

Here's a clear and structured summary of resources and videos to learn electricity and electromagnetism:

**Online Courses and Lectures**

1. MIT OpenCourseWare: "Physics II: Electricity and Magnetism"
2. IOPSpark (Unlimited Access to Physics Teaching Resources): Remote teaching resources for electricity and magnetism
3. MITx: Maxwell's Equations, electromagnetic radiation, and electromagnetic fields

**Videos**

1. The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism ( online edition)
2. AP Physics C: Electricity and Magnetism Full Course/Full Walkthrough (YouTube)
3. Khan Academy videos on electricity and magnetism
4. JoVE's science videos on electromagnetism

**Textbooks and Resources**

1. MIT OpenCourseWare textbook: Maxwell’s equations, in addition to describing this behavior, also describe electromagnetic radiation.
2. Becker/Sauter edition of Abraham/Becker (follow-up edition) for advanced undergraduate level study
3. Dover book: "Physics: Tutoring, Video Lessons, Co

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [ELECTROMAGNETISM for Kids⚡🧲 What are Electromagnets? 🔌 ](https://www.youtube.com/watch?v=tHIchO1pbFA) | Smile and Learn - English | 4:00 | 216,517 | `tHIchO1pbFA` |
| 2 | [Teach Yourself Physics from SCRATCH. | Foundations 1.1 ](https://www.youtube.com/watch?v=4c02Njfh2Zk) | Mike's Channel | 4:43 | 60,719 | `4c02Njfh2Zk` |
| 3 | [What is Light? Maxwell and the Electromagnetic Spectrum](https://www.youtube.com/watch?v=pj_ya0e20vE) | Professor Dave Explains | 3:56 | 1,384,639 | `pj_ya0e20vE` |
| 4 | [Physics for Absolute Beginners](https://www.youtube.com/watch?v=iE-M56CVJx4) | The Math Sorcerer | 13:06 | 386,835 | `iE-M56CVJx4` |
| 5 | [Electricity for Kids | What is Electricity? Where does ](https://www.youtube.com/watch?v=Dx3RpXdJw2k) | Learn Bright | 13:54 | 3,810,675 | `Dx3RpXdJw2k` |
| 6 | [Want to study physics? Read these 10 books](https://www.youtube.com/watch?v=p9s2fBYA4fU) | Simon Clark | 14:16 | 2,286,423 | `p9s2fBYA4fU` |


Video IDs:
  tHIchO1pbFA   https://www.youtube.com/watch?v=tHIchO1pbFA
  Educational video for children that talks about electromagnetism and electromagnets. Electricity is a physical

  4c02Njfh2Zk   https://www.youtube.com/watch?v=4c02Njfh2Zk
  Beyond belief so what I want you to do in this course is follow with me this is a textbook called physics by c

  pj_ya0e20vE   https://www.youtube.com/watch?v=pj_ya0e20vE
  Up until a couple centuries ago, we had no idea what light is. It seems like magic, no? But there is no magic 

  iE-M56CVJx4   https://www.youtube.com/watch?v=iE-M56CVJx4
  This video will show you some books you can use to help get started with physics. Do you have any other recomm

  Dx3RpXdJw2k   https://www.youtube.com/watch?v=Dx3RpXdJw2k
  NOTE: We would like to correct an error in this video. Birds do not get electrocuted when resting on power lin

  p9s2fBYA4fU   https://www.youtube.com/watch?v=p9s2fBYA4fU
  Books for physics students! Popular science books and te

Here's a clear and structured summary of resources and videos to learn electricity and electromagnetism:

**Online Courses and Lectures**

1. MIT OpenCourseWare: "Physics II: Electricity and Magnetism"
2. IOPSpark (Unlimited Access to Physics Teaching Resources): Remote teaching resources for electricity and magnetism
3. MITx: Maxwell's Equations, electromagnetic radiation, and electromagnetic fields

**Videos**

1. The Feynman Lectures on Physics Vol. II Ch. 1: Electromagnetism ( online edition)
2. AP Physics C: Electricity and Magnetism Full Course/Full Walkthrough (YouTube)
3. Khan Academy videos on electricity and magnetism
4. JoVE's science videos on electromagnetism

**Textbooks and Resources**

1. MIT OpenCourseWare textbook: Maxwell’s equations, in addition to describing this behavior, also describe electromagnetic radiation.
2. Becker/Sauter edition of Abraham/Becker (follow-up edition) for advanced undergraduate level study
3. Dover book: "Physics: Tutoring, Video Lessons, Courses, Problems & Practice" for AP Physics C: E and M skills

**Websites and Forums**

1. Physics Forums for self-study and discussion on electricity and magnetism
2. Khan Academy website with various electromagnetism resources


All data stored under search_id = 6


### Browse stored data

In [12]:
def db_history(limit=20):
    """Show all past searches."""
    with get_conn() as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("""
                SELECT s.id, s.topic, s.search_type, s.created_at,
                       COUNT(DISTINCT w.id) AS web_count,
                       COUNT(DISTINCT y.id) AS yt_count,
                       COUNT(DISTINCT r.id) AS report_count
                FROM searches s
                LEFT JOIN web_results    w ON w.search_id = s.id
                LEFT JOIN youtube_results y ON y.search_id = s.id
                LEFT JOIN reports        r ON r.search_id = s.id
                GROUP BY s.id ORDER BY s.created_at DESC LIMIT %s
            """, (limit,))
            rows = cur.fetchall()
    if not rows:
        print("No searches stored yet.")
        return []
    headers = "| id | topic | type | web | yt | reports | created |"
    sep     = "|----|-------|------|-----|----|---------|---------|"
    lines = [headers, sep]
    for r in rows:
        ts = r["created_at"].strftime("%Y-%m-%d %H:%M")
        lines.append(f"| {r['id']} | {r['topic'][:40]} | {r['search_type']} | {r['web_count']} | {r['yt_count']} | {r['report_count']} | {ts} |")
    display(Markdown("\n".join(lines)))
    return rows

def db_get_report(search_id):
    """Retrieve and display a stored report."""
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT content FROM reports WHERE search_id=%s ORDER BY id DESC LIMIT 1", (search_id,))
            row = cur.fetchone()
    if row:
        display(Markdown(row[0]))
    else:
        print(f"No report found for search_id={search_id}")

def db_get_youtube(search_id):
    """Retrieve and display stored YouTube results."""
    with get_conn() as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute("SELECT * FROM youtube_results WHERE search_id=%s", (search_id,))
            rows = cur.fetchall()
    if not rows:
        print(f"No YouTube results for search_id={search_id}")
        return
    videos = [dict(r) for r in rows]
    # remap keys for display_youtube_results
    for v in videos:
        v["id"]    = v["video_id"]
        v["desc"]  = v.get("description", "")
        v["thumb"] = v.get("thumbnail", "")
    display_youtube_results(videos)

print("DB viewer functions ready.")

DB viewer functions ready.


In [13]:
# View all past searches
db_history()

| id | topic | type | web | yt | reports | created |
|----|-------|------|-----|----|---------|---------|
| 6 | i want to learn physics and electromagne | combined | 10 | 6 | 1 | 2026-03-15 22:25 |
| 5 | i want to learn physics and electromagne | combined | 10 | 6 | 1 | 2026-03-15 21:58 |
| 4 | i want to learn about Machine Learning | combined | 10 | 6 | 1 | 2026-03-15 00:35 |
| 3 | i want to learn about quantum computing  | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 2 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 1 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:31 |

[RealDictRow([('id', 6),
              ('topic',
               'i want to learn physics and electromagnetism give me resources and videos'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 22, 25, 31, 379754, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 5),
              ('topic', 'i want to learn physics and electromagnetism'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 21, 58, 2, 439546, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 4),
              ('topic', 'i want to learn about Machine Learning'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 35, 57

In [35]:
# Load a saved report by search_id
db_get_report(1)

**Quantum Computing in 2024: Breakthroughs, Challenges, and What's Ahead**

Quantum computing has made significant progress in 2023, but as we move into 2024, several challenges must be addressed to realize its full potential. Here's a summary of the current state of quantum computing:

**Breakthroughs:**

1. **Advancements in Hardware:** IBM and other companies have developed new hardware designs that improve the performance and efficiency of quantum computers.
2. **Software Evolution:** Quantum algorithms are being developed and optimized for practical applications, such as cryptography and optimization problems.
3. **Cybersecurity Impacts:** Quantum computing has the potential to revolutionize cybersecurity by breaking certain types of encryption.

**Challenges:**

1. **Scalability:** Current quantum computers are not yet scalable enough to solve complex problems efficiently.
2. **Noise and Error Correction:** Maintaining control over quantum fluctuations (noise) is a significant challenge, and developing robust error correction methods is essential.
3. **Interpretation of Results:** Quantum computing generates vast amounts of data, which must be interpreted correctly for practical applications.

**Future Trends:**

1. **Quantum-AI Symbiosis:** Researchers are exploring the integration of quantum computers with artificial intelligence (AI) to drive advancements in fields like optimization and machine learning.
2. **Commercial Adoption:** Expect significant commercial adoption of quantum computing in 2024, driven by increased investment, partnerships, and public awareness.
3. **Roadmaps and Development Paths:** Multiple players are announcing new development paths or updating existing ones, solidifying the year's roadmap for quantum computing.

**Key Players:**

1. IBM
2. Google (Bostock Lab)
3. Rigetti Computing
4. Microsoft Quantum Development Kit

In summary, 2024 promises to be a transformative year for quantum computing, with significant breakthroughs in hardware, software, and challenges that must be addressed. As the field continues to evolve, expect exciting developments in areas like AI symbiosis, commercial adoption, and scalability.

In [36]:
# Load saved YouTube videos by search_id
db_get_youtube(1)

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Quantum Computing 2024 Update](https://www.youtube.com/watch?v=tIJuyb5DWUw) | ExplainingComputers | 15:41 | 92,343 | `tIJuyb5DWUw` |
| 2 | [Companies, countries battle to develop quantum computer](https://www.youtube.com/watch?v=K4ssT6Dzmnw) | 60 Minutes | 13:15 | 2,493,889 | `K4ssT6Dzmnw` |
| 3 | [Quantum Computers Explained: How Quantum Computing Work](https://www.youtube.com/watch?v=B3U1NDUiwSA) | Science ABC | 5:41 | 733,457 | `B3U1NDUiwSA` |
| 4 | [Quantum Computing: Where We Are and Where We’re Headed ](https://www.youtube.com/watch?v=9XB-LsfpvCU) | NVIDIA Developer | 123:20 | 230,803 | `9XB-LsfpvCU` |
| 5 | [Quantum Computing 2025 Update](https://www.youtube.com/watch?v=cDt7o-OmBWI) | ExplainingComputers | 17:05 | 61,948 | `cDt7o-OmBWI` |
| 6 | [Michio Kaku: Quantum computing is the next revolution](https://www.youtube.com/watch?v=qQviI1d_hFA) | Big Think | 11:18 | 3,350,527 | `qQviI1d_hFA` |


Video IDs:
  tIJuyb5DWUw   https://www.youtube.com/watch?v=tIJuyb5DWUw
  Quantum computing developments over the past 12 months, including progress towards fault-tolerant hardware, an

  K4ssT6Dzmnw   https://www.youtube.com/watch?v=K4ssT6Dzmnw
  Companies and countries are in a race to develop quantum computers. The machines could revolutionize problem-s

  B3U1NDUiwSA   https://www.youtube.com/watch?v=B3U1NDUiwSA
  Quantum computers use the principles of quantum mechanics to process information in ways that classical comput

  9XB-LsfpvCU   https://www.youtube.com/watch?v=9XB-LsfpvCU
  NVIDIA founder and CEO Jensen Huang hosts industry leaders from Alice & Bob, Atom Computing, AWS, D-Wave, Infl

  cDt7o-OmBWI   https://www.youtube.com/watch?v=cDt7o-OmBWI
  Quantum computing developments over the past 12 months, including Google Willow, IBM Majorana 1, and innovatio

  qQviI1d_hFA   https://www.youtube.com/watch?v=qQviI1d_hFA
  Become a Big Think member to unlock expert classes, prem

In [38]:
# ---- RUN WITH DB STORAGE ----
DB_TOPIC = "i want to learn about quantum computing breakthroughs in 2024"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

**Quantum Computing Breakthroughs in 2024**

In 2024, significant advancements have been made in quantum computing, transforming the field from theory to tangible progress. Some of the key breakthroughs include:

1. **Improved Quantum Engineering**: Researchers have developed new methods for improving quantum systems, enabling more stable and reliable quantum computing.
2. **Logical Qubits**: Breakthroughs have led to improvements in logical qubits, which are the fundamental building blocks of quantum computers. These improvements have increased the reliability and error correction capabilities of quantum computers.
3. **Noise Resistance**: Researchers have demonstrated significant improvements in noise resistance, enabling quantum computers to operate more reliably in real-world settings.

**Key Developments:**

* Google's Willow chip
* 50 logical qubits
* Advances in quantum error correction

**Hybrid Approaches**: Research groups have explored the use of hybrid approaches that combi

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Quantum Computers Explained: How Quantum Computing Work](https://www.youtube.com/watch?v=B3U1NDUiwSA) | Science ABC | 5:41 | 733,457 | `B3U1NDUiwSA` |
| 2 | [Biggest Breakthroughs in Computer Science: 2024](https://www.youtube.com/watch?v=fTMMsreAqX0) | Quanta Magazine | 10:47 | 527,759 | `fTMMsreAqX0` |
| 3 | [Quantum Computing Breakthroughs of 2024: 10 Ways It Wil](https://www.youtube.com/watch?v=RtYchdRl7Bo) | SoSF | 24:17 | 204 | `RtYchdRl7Bo` |
| 4 | [Michio Kaku: Quantum computing is the next revolution](https://www.youtube.com/watch?v=qQviI1d_hFA) | Big Think | 11:18 | 3,350,527 | `qQviI1d_hFA` |
| 5 | [Decoding the Universe: Quantum | Full Documentary | NOV](https://www.youtube.com/watch?v=t06aTX9jM34) | NOVA PBS Official | 53:58 | 7,486,513 | `t06aTX9jM34` |
| 6 | [New Quantum Physics Discoveries 2024](https://www.youtube.com/watch?v=g5qKExdt4wo) | SpaceXVideos | 19:40 | 1,157 | `g5qKExdt4wo` |


Video IDs:
  B3U1NDUiwSA   https://www.youtube.com/watch?v=B3U1NDUiwSA
  Quantum computers use the principles of quantum mechanics to process information in ways that classical comput

  fTMMsreAqX0   https://www.youtube.com/watch?v=fTMMsreAqX0
  The year's biggest breakthroughs in computer science included a new understanding of what's going on in large 

  RtYchdRl7Bo   https://www.youtube.com/watch?v=RtYchdRl7Bo
  Voice and cover generated by AI The drafting and researching are done by me, but with AI involvement Video edi

  qQviI1d_hFA   https://www.youtube.com/watch?v=qQviI1d_hFA
  Become a Big Think member to unlock expert classes, premium print issues, exclusive events and more: ...

  t06aTX9jM34   https://www.youtube.com/watch?v=t06aTX9jM34
  Dive into the universe at the tiniest – and weirdest – of scales. Official Website: https://to.pbs.org/3CkDYDR

  g5qKExdt4wo   https://www.youtube.com/watch?v=g5qKExdt4wo
  Join us on a fascinating journey through the world of quantum 

**Quantum Computing Breakthroughs in 2024**

In 2024, significant advancements have been made in quantum computing, transforming the field from theory to tangible progress. Some of the key breakthroughs include:

1. **Improved Quantum Engineering**: Researchers have developed new methods for improving quantum systems, enabling more stable and reliable quantum computing.
2. **Logical Qubits**: Breakthroughs have led to improvements in logical qubits, which are the fundamental building blocks of quantum computers. These improvements have increased the reliability and error correction capabilities of quantum computers.
3. **Noise Resistance**: Researchers have demonstrated significant improvements in noise resistance, enabling quantum computers to operate more reliably in real-world settings.

**Key Developments:**

* Google's Willow chip
* 50 logical qubits
* Advances in quantum error correction

**Hybrid Approaches**: Research groups have explored the use of hybrid approaches that combine classical computing, artificial intelligence (AI), and quantum hardware to tackle complex problems more effectively.

**Future Predictions:**

* Google's Willow processor expected to be operational by 2025
* Harvard's 48-logical-qubit processor expected to be operational in 2026

**Industry Implications:**

* Quantum computers may potentially break current encryption methods, necessitating advancements in cryptographic techniques.
* Quantum computers have the potential to revolutionize various fields, including medical science, machine learning (ML), environmental sciences, and other areas of advancement.

Overall, 2024 has seen significant progress in quantum computing, with breakthroughs that are transforming the field from theory to tangible progress. As research continues to advance, we can expect even more exciting developments in the years to come.


All data stored under search_id = 3


In [39]:
db_history()

| id | topic | type | web | yt | reports | created |
|----|-------|------|-----|----|---------|---------|
| 3 | i want to learn about quantum computing  | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 2 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 1 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:31 |

[RealDictRow([('id', 3),
              ('topic',
               'i want to learn about quantum computing breakthroughs in 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 44, 35126, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 2),
              ('topic', 'quantum computing 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 20, 763229, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 1),
              ('topic', 'quantum computing 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 31, 59, 289279, tzinfo=datetime.timezone.utc)),
          

In [40]:
DB_TOPIC = "i want to learn about Machine Learning"
# -----------------------------

sid = db_full_research(DB_TOPIC)
print(f"\nAll data stored under search_id = {sid}")

Here's a clear structured answer on "How To Learn Machine Learning in 2025" based on the provided resources:

**Step-by-Step Guide to Learning Machine Learning**

1. **Prerequisites**: Start with essential mathematical and programming concepts, such as linear algebra, calculus, probability, and computer science fundamentals.
2. **Setup Environment**: Install necessary tools and libraries, including Python, scikit-learn, TensorFlow, or PyTorch.
3. **Key Concepts**: Learn about machine learning algorithms, data preprocessing, model evaluation, and interpretation.
4. **Algorithms**: Study popular algorithms like linear regression, decision trees, clustering, and neural networks.
5. **Real-World Projects**: Participate in machine learning competitions, build projects, or work on personal initiatives to apply concepts learned.
6. **Certification Programs**: Consider obtaining certifications from organizations like Google, IBM, or edX to demonstrate expertise.

**Recommended Resources**

* *

| # | Title | Channel | Duration | Views | ID |
|---|-------|---------|----------|-------|----|
| 1 | [Machine Learning Explained in 100 Seconds](https://www.youtube.com/watch?v=PeMlggyqz0Y) | Fireship | 2:35 | 1,020,680 | `PeMlggyqz0Y` |
| 2 | [Machine Learning | What Is Machine Learning? | Introduc](https://www.youtube.com/watch?v=ukzFI9rgwfU) | Simplilearn | 7:52 | 5,372,393 | `ukzFI9rgwfU` |
| 3 | [The Complete Machine Learning Roadmap](https://www.youtube.com/watch?v=7IgVGSaQPaw) | Programming with Mosh | 5:25 | 944,331 | `7IgVGSaQPaw` |
| 4 | [Learn Machine Learning Like a GENIUS and Not Waste Time](https://www.youtube.com/watch?v=qNxrPri1V0I) | Infinite Codes | 15:03 | 1,193,951 | `qNxrPri1V0I` |
| 5 | [Learn Machine Learning Like a GENIUS and Not Waste Time](https://www.youtube.com/watch?v=2wTRQqoDM1A) | Sahil & Sarra | 10:33 | 288,833 | `2wTRQqoDM1A` |
| 6 | [Machine Learning is Probably Not a Good Career for You](https://www.youtube.com/watch?v=-8qzzN7CWxA) | the data janitor | 1:38 | 119,037 | `-8qzzN7CWxA` |


Video IDs:
  PeMlggyqz0Y   https://www.youtube.com/watch?v=PeMlggyqz0Y
  Machine Learning is the process of teaching a computer how perform a task with out explicitly programming it. 

  ukzFI9rgwfU   https://www.youtube.com/watch?v=ukzFI9rgwfU
  "️   Michigan Engineering - Professional Certificate in AI and Machine Learning ...

  7IgVGSaQPaw   https://www.youtube.com/watch?v=7IgVGSaQPaw
  Go from zero to a machine learning engineer in 12 months. This step-by-step roadmap covers the essential skill

  qNxrPri1V0I   https://www.youtube.com/watch?v=qNxrPri1V0I
  Learn Machine Learning Like a GENIUS and Not Waste Time ######################################### I just start

  2wTRQqoDM1A   https://www.youtube.com/watch?v=2wTRQqoDM1A
  Get Coursera Plus for 40% OFF 3 months: https://imp.i384100.net/c/3552395/3391150/14726 ▻ For more content lik

  -8qzzN7CWxA   https://www.youtube.com/watch?v=-8qzzN7CWxA
  This isn't a career most will succeed in.



Here's a clear structured answer on "How To Learn Machine Learning in 2025" based on the provided resources:

**Step-by-Step Guide to Learning Machine Learning**

1. **Prerequisites**: Start with essential mathematical and programming concepts, such as linear algebra, calculus, probability, and computer science fundamentals.
2. **Setup Environment**: Install necessary tools and libraries, including Python, scikit-learn, TensorFlow, or PyTorch.
3. **Key Concepts**: Learn about machine learning algorithms, data preprocessing, model evaluation, and interpretation.
4. **Algorithms**: Study popular algorithms like linear regression, decision trees, clustering, and neural networks.
5. **Real-World Projects**: Participate in machine learning competitions, build projects, or work on personal initiatives to apply concepts learned.
6. **Certification Programs**: Consider obtaining certifications from organizations like Google, IBM, or edX to demonstrate expertise.

**Recommended Resources**

* **GeeksforGeeks**: A comprehensive guide covering various aspects of machine learning from the basics to advanced topics.
* **IBM Machine Learning Guide**: Offers a structured approach to learning machine learning with modules and tutorials.
* **Google for Developers Machine Learning Crash Course**: Self-contained modules covering key concepts, algorithms, and projects.
* **DataCamp's Machine Learning in 2026**: A step-by-step guide to learning machine learning from scratch.

**Career Opportunities**

Machine learning is a rapidly growing field with various career paths, including:

* Data Scientist
* AI/ML Engineer
* Researcher
* Business Analyst

**Conclusion**

Learning machine learning requires dedication and persistence. By following this step-by-step guide, exploring recommended resources, and pursuing certification programs, individuals can develop the skills needed to succeed in this exciting field.


All data stored under search_id = 4


In [41]:
db_history()

| id | topic | type | web | yt | reports | created |
|----|-------|------|-----|----|---------|---------|
| 4 | i want to learn about Machine Learning | combined | 10 | 6 | 1 | 2026-03-15 00:35 |
| 3 | i want to learn about quantum computing  | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 2 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:33 |
| 1 | quantum computing 2024 | combined | 10 | 6 | 1 | 2026-03-15 00:31 |

[RealDictRow([('id', 4),
              ('topic', 'i want to learn about Machine Learning'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 35, 57, 161382, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 3),
              ('topic',
               'i want to learn about quantum computing breakthroughs in 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 44, 35126, tzinfo=datetime.timezone.utc)),
              ('web_count', 10),
              ('yt_count', 6),
              ('report_count', 1)]),
 RealDictRow([('id', 2),
              ('topic', 'quantum computing 2024'),
              ('search_type', 'combined'),
              ('created_at',
               datetime.datetime(2026, 3, 15, 0, 33, 20, 763229, tzinfo=datetime.timezone.u

----------------------------


## 9. LearnFlow — Personalised Learning Graph

A full graph-based adaptive learning system running entirely on Ollama + DuckDuckGo.

```
[START]
   │
   ▼
[generate_assessment]  ── 8-question diagnostic quiz
   │
   ▼
[wait_for_assessment]  ── INTERRUPT: collect user answers
   │
   ▼
[analyze_level]        ── score answers → level + gaps
   │
   ▼
[research_resources]   ── web search for relevant material
   │
   ▼
[build_roadmap]        ── personalised 4-6 module plan
   │
   ▼
[generate_content]  ◄──────────────────────────────────────┐
   │                                                        │ (failed)
   ▼                                                        │
[wait_for_quiz_req]    ── INTERRUPT: 'quiz' | 'more'        │
   │                                                        │
   ▼                                                        │
[generate_quiz]        ── 5 quiz questions from content     │
   │                                                        │
   ▼                                                        │
[wait_for_quiz_ans]    ── INTERRUPT: collect quiz answers   │
   │                                                        │
   ▼                                                        │
[evaluate_quiz]        ── score, feedback, save to DB       │
   │                                                        │
   ▼                                                        │
[quiz_router] ── score ≥ 70 → [advance] ── more → generate_content ──┘
                  score < 70 ────────────────────────────────────────┘
                                           done → [END] 🏆
```


In [14]:

import json
from dataclasses import dataclass, field
from typing import Optional

# ── LearnFlow State ───────────────────────────────────────────────────────────

@dataclass
class LearnFlowState:
    topic:                str   = ""
    assessment_questions: list  = field(default_factory=list)
    assessment_answers:   list  = field(default_factory=list)
    level:                str   = ""          # beginner | intermediate | advanced
    gaps:                 list  = field(default_factory=list)
    resources:            list  = field(default_factory=list)
    roadmap:              list  = field(default_factory=list)
    current_module:       int   = 0
    content:              str   = ""
    quiz:                 list  = field(default_factory=list)
    quiz_answers:         list  = field(default_factory=list)
    quiz_score:           float = 0.0
    attempts:             int   = 0
    completed:            bool  = False
    search_id:            Optional[int] = None


# ── Minimal graph engine (LangGraph-style) ────────────────────────────────────

END = "__END__"


class LearnFlowGraph:
    def __init__(self):
        self.nodes              = {}
        self.edges              = {}
        self.conditional_edges  = {}
        self._entry             = None

    def add_node(self, name, fn):
        self.nodes[name] = fn

    def set_entry_point(self, name):
        self._entry = name

    def add_edge(self, src, dst):
        self.edges[src] = dst

    def add_conditional_edges(self, src, condition_fn, mapping):
        self.conditional_edges[src] = (condition_fn, mapping)

    def run(self, state):
        current = self._entry
        while current and current != END:
            print(f"\n{'─'*50}\n[GRAPH] ▶  {current}\n{'─'*50}")
            state = self.nodes[current](state)
            if current in self.conditional_edges:
                cond_fn, mapping = self.conditional_edges[current]
                current = mapping.get(cond_fn(state), END)
            else:
                current = self.edges.get(current)
        print("\n🏆  LearnFlow complete!")
        return state


print("LearnFlowState & LearnFlowGraph defined.")


LearnFlowState & LearnFlowGraph defined.


In [15]:

# ── Nodes: assessment, level analysis, research, roadmap ──────────────────────

def generate_assessment(state):
    """Generate 8-question diagnostic quiz."""
    print(f"[1/4] Generating diagnostic assessment for: {state.topic}")
    prompt = f"""Create a diagnostic quiz of exactly 8 questions to assess knowledge of: "{state.topic}"

Rules:
- Mix difficulty: 2 easy, 4 medium, 2 hard
- Include multiple-choice (A/B/C/D) and short-answer questions
- Each question targets a distinct concept or skill

Return a JSON array — nothing else:
[{{"q":"question text", "type":"mcq"|"short",
   "options":["A)...","B)...","C)...","D)..."]|null,
   "answer":"correct answer", "concept":"concept tested"}}]"""

    raw   = llm(prompt)
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    try:
        questions = json.loads(match.group()) if match else []
    except Exception:
        questions = []
    if not questions:
        questions = [
            {"q": f"What is {state.topic}?",          "type": "short", "options": None, "answer": "", "concept": "definition"},
            {"q": f"Name a key application of {state.topic}.", "type": "short", "options": None, "answer": "", "concept": "application"},
        ]
    state.assessment_questions = questions[:8]
    print(f"  Generated {len(state.assessment_questions)} questions.")
    return state


def wait_for_assessment(state):
    """INTERRUPT: Display questions and collect user answers."""
    print(f"\n{'='*60}\n📋  DIAGNOSTIC ASSESSMENT — {state.topic}\n{'='*60}\n")
    answers = []
    for i, q in enumerate(state.assessment_questions, 1):
        print(f"Q{i}: {q['q']}")
        if q.get("options"):
            for opt in q["options"]:
                print(f"   {opt}")
            ans = input("   Your answer (A/B/C/D): ").strip().upper()
        else:
            ans = input("   Your answer: ").strip()
        answers.append(ans)
        print()
    state.assessment_answers = answers
    print("✅  Assessment submitted.")
    return state


def analyze_level(state):
    """Score answers, detect level and knowledge gaps."""
    print("[2/4] Analysing your level...")
    qa_pairs = "\n".join(
        f"Q{i+1}: {q['q']}\nGiven: {a}\nExpected: {q.get('answer','N/A')}\nConcept: {q.get('concept','')}"
        for i, (q, a) in enumerate(zip(state.assessment_questions, state.assessment_answers))
    )
    prompt = f"""Evaluate quiz answers for topic "{state.topic}":
{qa_pairs}

Return JSON — nothing else:
{{"level":"beginner|intermediate|advanced",
  "gaps":["gap1","gap2","gap3"],
  "score":0-100}}"""

    raw   = llm(prompt)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    try:
        result      = json.loads(match.group()) if match else {}
        state.level = result.get("level", "beginner")
        state.gaps  = result.get("gaps", [f"fundamentals of {state.topic}"])
    except Exception:
        state.level = "beginner"
        state.gaps  = [f"fundamentals of {state.topic}"]

    print(f"  Level : {state.level.upper()}")
    print(f"  Gaps  : {state.gaps}")
    return state


def research_resources(state):
    """Web search for learning resources tailored to level and gaps."""
    print("[3/4] Researching learning resources...")
    queries = [f"{state.topic} {state.level} tutorial"] + \
              [f"{g} explained" for g in state.gaps[:2]]

    seen, unique = set(), []
    for q in queries:
        for r in search_web(q, max_results=4):
            if r["url"] and r["url"] not in seen:
                seen.add(r["url"])
                unique.append(r)
        time.sleep(0.3)

    state.resources = unique[:12]

    # ── Render resources as Markdown ─────────────────────────────────────────
    lines = [
        f"## 🔍 Found {len(state.resources)} Learning Resources",
        f"",
        f"| # | Title | Snippet |",
        f"|---|-------|---------|",
    ]
    for i, r in enumerate(state.resources, 1):
        title   = f"[{r['title'][:55]}]({r['url']})" if r.get("url") else r.get("title", "")[:55]
        snippet = r.get("snippet", "")[:90].replace("|", "\|")
        lines.append(f"| {i} | {title} | {snippet}… |")
    display(Markdown("\n".join(lines)))
    return state


def build_roadmap(state):
    """Generate a personalised module plan."""
    print("[4/4] Building personalised roadmap...")
    resource_titles = "\n".join(f"- {r['title']}: {r['snippet'][:100]}" for r in state.resources[:6])
    gaps_str        = "\n".join(f"- {g}" for g in state.gaps)

    prompt = f"""Create a personalised learning roadmap:
Topic : {state.topic}
Level : {state.level}
Gaps  :
{gaps_str}

Available resources:
{resource_titles}

Design 4-6 progressive modules. Return a JSON array — nothing else:
[{{"module":1, "title":"...", "objective":"...",
   "concepts":["c1","c2"], "duration":"30 min"}}]"""

    raw   = llm(prompt)
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    try:
        roadmap = json.loads(match.group()) if match else []
    except Exception:
        roadmap = []
    if not roadmap:
        roadmap = [
            {"module": 1, "title": f"Intro to {state.topic}",     "objective": "Understand basics",   "concepts": ["overview"],  "duration": "30 min"},
            {"module": 2, "title": f"Core {state.topic} concepts", "objective": "Build core knowledge","concepts": ["core"],      "duration": "45 min"},
            {"module": 3, "title": f"Applied {state.topic}",       "objective": "Practical skills",    "concepts": ["practice"],  "duration": "60 min"},
        ]

    state.roadmap        = roadmap
    state.current_module = 0

    # ── Render roadmap as a rich Markdown table ──────────────────────────────
    level_badge = {"beginner": "🟢 Beginner", "intermediate": "🟡 Intermediate", "advanced": "🔴 Advanced"}.get(state.level, state.level)
    lines = [
        f"## 📚 Your Personalised Learning Roadmap",
        f"",
        f"**Topic:** {state.topic}  ·  **Level:** {level_badge}",
        f"",
        f"**Knowledge gaps addressed:**",
    ]
    for g in state.gaps:
        lines.append(f"- {g}")
    lines += [
        f"",
        f"---",
        f"",
        f"| # | Module | Objective | Concepts | Duration |",
        f"|---|--------|-----------|----------|----------|",
    ]
    for m in roadmap:
        concepts = ", ".join(m.get("concepts", []))
        lines.append(
            f"| {m.get('module','?')} | **{m.get('title','?')}** "
            f"| {m.get('objective','')} | {concepts} | ⏱ {m.get('duration','?')} |"
        )
    lines += [
        f"",
        f"---",
        f"*{len(roadmap)} modules · estimated total: "
        + _total_duration(roadmap) + "*",
    ]
    display(Markdown("\n".join(lines)))
    return state


def _total_duration(roadmap):
    """Sum durations like '30 min', '1 hour' into a readable string."""
    total_min = 0
    for m in roadmap:
        d = m.get("duration", "")
        nums = re.findall(r"\d+", d)
        if nums:
            mins = int(nums[0])
            if "hour" in d:
                mins *= 60
            total_min += mins
    if total_min >= 60:
        return f"{total_min // 60}h {total_min % 60:02d}min"
    return f"{total_min} min"


print("Assessment, analysis, research & roadmap nodes defined.")


Assessment, analysis, research & roadmap nodes defined.


<>:108: SyntaxWarning: invalid escape sequence '\|'
<>:108: SyntaxWarning: invalid escape sequence '\|'
/var/folders/dp/ptmdptw102qfsw6n40jzf7zw0000gn/T/ipykernel_67690/3441384547.py:108: SyntaxWarning: invalid escape sequence '\|'
  snippet = r.get("snippet", "")[:90].replace("|", "\|")


In [16]:

# ── Nodes: content, quiz, evaluation, routing ─────────────────────────────────

def generate_content(state):
    """Generate lesson content for the current module."""
    module = state.roadmap[state.current_module]
    total  = len(state.roadmap)
    print(f"Generating content — Module {state.current_module + 1}/{total}: {module.get('title')}")

    context = "\n\n".join(
        f"{r['title']}: {r['snippet']}"
        for r in state.resources[:5] if r.get("snippet")
    )
    system = "You are an expert tutor. Write clear, engaging lesson content with concrete examples."
    prompt = f"""Write a comprehensive lesson for:
Module: {module.get('title')}
Objective: {module.get('objective')}
Concepts: {', '.join(module.get('concepts', []))}
Learner level: {state.level}

Reference material:
{context[:2000]}

Structure with: ## Overview, ## Key Concepts (with examples), ## Practical Application, ## Summary
Use markdown formatting."""

    state.content  = llm(prompt, system=system)
    state.attempts = 0  # reset per module

    print(f"\n{'='*60}\n📖  MODULE {state.current_module + 1}: {module.get('title')}\n{'='*60}")
    display(Markdown(state.content))
    return state


def wait_for_quiz_req(state):
    """INTERRUPT: Wait for user to request the quiz or re-read the lesson."""
    print("\n" + "─"*60)
    print("  Type  'quiz'  to test your knowledge.")
    print("  Type  'more'  to re-read the content.")
    print("─"*60)
    while True:
        cmd = input("  Your choice: ").strip().lower()
        if cmd == "quiz":
            break
        elif cmd == "more":
            display(Markdown(state.content))
        else:
            print("  Please type 'quiz' or 'more'.")
    return state


def generate_quiz(state):
    """Generate 5 quiz questions from lesson content."""
    module = state.roadmap[state.current_module]
    print("Generating quiz questions...")
    prompt = f"""Create exactly 5 quiz questions to test understanding of:
Module: {module.get('title')}
Content: {state.content[:1500]}

Mix: 3 multiple-choice (A/B/C/D) + 2 short-answer.
Return a JSON array:
[{{"q":"...", "type":"mcq"|"short", "options":["A)...","B)...","C)...","D)..."]|null, "answer":"correct answer"}}]
Return ONLY valid JSON."""

    raw   = llm(prompt)
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    try:
        state.quiz = json.loads(match.group())[:5] if match else []
    except Exception:
        state.quiz = []
    if not state.quiz:
        state.quiz = [{"q": f"Summarise what you learned about {module.get('title')}.",
                       "type": "short", "options": None, "answer": ""}]
    return state


def wait_for_quiz_ans(state):
    """INTERRUPT: Display quiz and collect user answers."""
    module = state.roadmap[state.current_module]
    print(f"\n{'='*60}\n🧠  QUIZ — Module {state.current_module + 1}: {module.get('title')}\n{'='*60}\n")
    answers = []
    for i, q in enumerate(state.quiz, 1):
        print(f"Q{i}: {q['q']}")
        if q.get("options"):
            for opt in q["options"]:
                print(f"   {opt}")
            ans = input("   Your answer (A/B/C/D): ").strip().upper()
        else:
            ans = input("   Your answer: ").strip()
        answers.append(ans)
        print()
    state.quiz_answers = answers
    return state


def evaluate_quiz(state):
    """Score quiz and update knowledge tracking."""
    module = state.roadmap[state.current_module]
    print("Evaluating quiz...")
    qa_pairs = "\n".join(
        f"Q{i+1}: {q['q']}\nGiven: {a}\nCorrect: {q.get('answer','')}"
        for i, (q, a) in enumerate(zip(state.quiz, state.quiz_answers))
    )
    prompt = f"""Grade this quiz for module "{module.get('title')}":
{qa_pairs}

Partial credit for short-answer if the concept is correct.
Return JSON: {{"score": 0-100, "passed": true|false,
  "feedback": [{{"q":1, "correct":true|false, "explanation":"..."}}]}}
Passing threshold: 70. Return ONLY valid JSON."""

    raw   = llm(prompt)
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    try:
        result         = json.loads(match.group()) if match else {}
        state.quiz_score = float(result.get("score", 0))
        feedback         = result.get("feedback", [])
    except Exception:
        state.quiz_score = 50.0
        feedback         = []

    state.attempts += 1
    print(f"\n📊  Score: {state.quiz_score:.0f}/100  (attempt {state.attempts})")
    for fb in feedback:
        icon = "✅" if fb.get("correct") else "❌"
        print(f"   {icon} Q{fb.get('q','?')}: {fb.get('explanation','')[:90]}")

    try:
        _lf_save_result(state)
    except Exception:
        pass
    return state



# ── Node: suggest_resources ───────────────────────────────────────────────────

def suggest_resources(state):
    """After quiz evaluation, suggest YouTube videos + educational web resources."""
    module  = state.roadmap[state.current_module]
    topic   = module.get("title", state.topic)
    passed  = state.quiz_score >= 70
    level   = state.level

    # Build targeted queries
    base_query = f"{state.topic} {topic} {level}"
    edu_query  = f"{base_query} tutorial lecture"

    print(f"\n{'─'*60}")
    print(f"📚  RESOURCE SUGGESTIONS — {topic}")
    print(f"{'─'*60}")

    # ── YouTube videos ────────────────────────────────────────────────────────
    print("\n🎬  YouTube Videos:")
    try:
        videos = search_youtube(f"{base_query} explained", max_results=4)
        if videos:
            rows = [
                "| # | Title | Channel | Duration |",
                "|---|-------|---------|----------|",
            ]
            for i, v in enumerate(videos, 1):
                title = f"[{v['title'][:55]}]({v['url']})"
                rows.append(f"| {i} | {title} | {v['channel']} | {v['duration']} |")
            display(Markdown("\n".join(rows)))
        else:
            print("  No videos found.")
    except Exception as e:
        print(f"  [YouTube error: {e}]")

    # ── Educational websites ──────────────────────────────────────────────────
    print("\n🌐  Educational Resources (universities & learning platforms):")
    try:
        edu_sites = [
            "site:coursera.org OR site:edx.org OR site:khanacademy.org",
            "site:mit.edu OR site:stanford.edu OR site:ocw.mit.edu",
        ]
        all_web = []
        seen    = set()
        for site_filter in edu_sites:
            results = search_web(f"{edu_query} {site_filter}", max_results=3)
            for r in results:
                if r["url"] not in seen:
                    seen.add(r["url"])
                    all_web.append(r)
            time.sleep(0.2)

        # Fallback: generic educational search if above returns nothing
        if not all_web:
            all_web = search_web(f"{edu_query} course lecture notes pdf", max_results=5)

        if all_web:
            lines = []
            for r in all_web[:6]:
                title   = r.get("title", r["url"])
                snippet = r.get("snippet", "")[:100]
                lines.append(f"- **[{title}]({r['url']})**  \n  {snippet}")
            display(Markdown("\n\n".join(lines)))
        else:
            print("  No resources found.")
    except Exception as e:
        print(f"  [Web error: {e}]")

    # ── Context-aware tip ─────────────────────────────────────────────────────
    if passed:
        print(f"\n💡  Great job! These resources will deepen your understanding before the next module.")
    else:
        print(f"\n💡  Review these before retrying — they cover the gaps detected in your quiz.")

    print(f"{'─'*60}\n")
    return state


# ── Routing functions ─────────────────────────────────────────────────────────

def quiz_router(state):
    if state.quiz_score >= 70:
        print("  ✅  Passed!")
        return "passed"
    elif state.attempts >= 3:
        print("  ⚠️   Max attempts reached — advancing anyway.")
        return "passed"
    else:
        print(f"  ❌  Score {state.quiz_score:.0f} < 70. Reviewing content again…")
        return "failed"


def advance(state):
    state.current_module += 1
    print(f"  ➡️   Module {state.current_module + 1} / {len(state.roadmap)}")
    return state


def completion_router(state):
    if state.current_module >= len(state.roadmap):
        state.completed = True
        print("\n🏆  All modules completed!")
        return "done"
    return "more"


print("Content, quiz & routing nodes defined.")


Content, quiz & routing nodes defined.


In [17]:

# ── PostgreSQL schema for LearnFlow ──────────────────────────────────────────
def init_learnflow_db():
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                CREATE TABLE IF NOT EXISTS learnflow_sessions (
                    id         SERIAL PRIMARY KEY,
                    topic      TEXT NOT NULL,
                    level      TEXT,
                    gaps       JSONB,
                    roadmap    JSONB,
                    created_at TIMESTAMPTZ DEFAULT NOW()
                );
                CREATE TABLE IF NOT EXISTS learnflow_results (
                    id           SERIAL PRIMARY KEY,
                    session_id   INT REFERENCES learnflow_sessions(id) ON DELETE CASCADE,
                    module_idx   INT,
                    module_title TEXT,
                    score        FLOAT,
                    attempts     INT,
                    passed       BOOLEAN,
                    created_at   TIMESTAMPTZ DEFAULT NOW()
                );
            """)
        conn.commit()
    print("LearnFlow DB tables ready.")


def _lf_save_result(state):
    module = state.roadmap[state.current_module] if state.current_module < len(state.roadmap) else {}
    if state.search_id is None:
        with get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(
                    "INSERT INTO learnflow_sessions (topic, level, gaps, roadmap)"
                    " VALUES (%s,%s,%s,%s) RETURNING id",
                    (state.topic, state.level,
                     json.dumps(state.gaps), json.dumps(state.roadmap))
                )
                state.search_id = cur.fetchone()[0]
            conn.commit()

    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO learnflow_results"
                " (session_id, module_idx, module_title, score, attempts, passed)"
                " VALUES (%s,%s,%s,%s,%s,%s)",
                (state.search_id, state.current_module,
                 module.get("title", ""), state.quiz_score,
                 state.attempts, state.quiz_score >= 70)
            )
        conn.commit()


try:
    init_learnflow_db()
except Exception as e:
    print(f"[DB] Not available — results won't be persisted: {e}")


# ── Graph assembly ────────────────────────────────────────────────────────────
def build_learnflow_graph():
    g = LearnFlowGraph()

    g.add_node("generate_assessment", generate_assessment)
    g.add_node("wait_for_assessment", wait_for_assessment)
    g.add_node("analyze_level",       analyze_level)
    g.add_node("research_resources",  research_resources)
    g.add_node("build_roadmap",       build_roadmap)
    g.add_node("generate_content",    generate_content)
    g.add_node("wait_for_quiz_req",   wait_for_quiz_req)
    g.add_node("generate_quiz",       generate_quiz)
    g.add_node("wait_for_quiz_ans",   wait_for_quiz_ans)
    g.add_node("evaluate_quiz",       evaluate_quiz)
    g.add_node("advance",             advance)
    g.add_node("suggest_resources",   suggest_resources)

    g.set_entry_point("generate_assessment")

    # Linear edges
    g.add_edge("generate_assessment", "wait_for_assessment")
    g.add_edge("wait_for_assessment", "analyze_level")
    g.add_edge("analyze_level",       "research_resources")
    g.add_edge("research_resources",  "build_roadmap")
    g.add_edge("build_roadmap",       "generate_content")
    g.add_edge("generate_content",    "wait_for_quiz_req")
    g.add_edge("wait_for_quiz_req",   "generate_quiz")
    g.add_edge("generate_quiz",       "wait_for_quiz_ans")
    g.add_edge("wait_for_quiz_ans",   "evaluate_quiz")

    g.add_edge("evaluate_quiz",      "suggest_resources")

    # Conditional: quiz_router — passed → advance, failed → retry content
    g.add_conditional_edges("suggest_resources", quiz_router, {
        "passed": "advance",
        "failed": "generate_content",
    })

    # Conditional: completion_router — more → next module, done → END
    g.add_conditional_edges("advance", completion_router, {
        "more": "generate_content",
        "done": END,
    })

    return g


print("LearnFlow graph assembled. Run build_learnflow_graph() to start.")


LearnFlow DB tables ready.
LearnFlow graph assembled. Run build_learnflow_graph() to start.


In [ ]:
# ── RUN LEARNFLOW ────────────────────────────────────────────────────────────
LF_TOPIC = "i want to ML"   
# ─────────────────────────────────────────────────────────────────────────────

graph = build_learnflow_graph()
initial_state = LearnFlowState(topic=LF_TOPIC)
final_state = graph.run(initial_state)



──────────────────────────────────────────────────
[GRAPH] ▶  generate_assessment
──────────────────────────────────────────────────
[1/4] Generating diagnostic assessment for: i want to ML
  Generated 2 questions.

──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_assessment
──────────────────────────────────────────────────

📋  DIAGNOSTIC ASSESSMENT — i want to ML

Q1: What is i want to ML?

Q2: Name a key application of i want to ML.

✅  Assessment submitted.

──────────────────────────────────────────────────
[GRAPH] ▶  analyze_level
──────────────────────────────────────────────────
[2/4] Analysing your level...
  Level : BEGINNER
  Gaps  : ['fundamentals of i want to ML']

──────────────────────────────────────────────────
[GRAPH] ▶  research_resources
──────────────────────────────────────────────────
[3/4] Researching learning resources...
  Found 8 resources.

──────────────────────────────────────────────────
[GRAPH] ▶  build_roadmap
────────────────────

# Introduction to Machine Learning
=====================================

## What is Machine Learning?
---------------------------

Machine learning (ML) is a subset of artificial intelligence that involves training algorithms to make predictions or decisions based on data. It's a key component of many modern technologies, including computer vision, natural language processing, and predictive analytics.

## Key Concepts in Machine Learning
---------------------------------

### Classification

Classification is a type of supervised machine learning where the goal is to predict a categorical label or class for a given input. In classification problems, we have a set of training examples with their corresponding labels, and our task is to create a model that can make predictions on new, unseen data.

Example: Image Classification

*   Given an image, what type of object does it represent?
*   Train a model using a dataset of images with their respective objects (e.g., dogs, cats, buildings)

### Regression

Regression is a type of supervised machine learning where the goal is to predict a continuous value for a given input. In regression problems, we have a set of training examples with their corresponding values, and our task is to create a model that can make predictions on new data.

Example: Temperature Prediction

*   What temperature will it be tomorrow?
*   Train a model using a dataset of historical temperatures (e.g., current temperature vs. previous days)

## Applying Machine Learning
---------------------------

Machine learning has numerous applications in various fields, including:

### Healthcare

*   Predict patient outcomes (e.g., disease diagnosis)
*   Personalize treatment plans based on individual characteristics

### Finance

*   Forecast stock prices
*   Identify potential risks and opportunities for investment

### Autonomous Vehicles

*   Detect obstacles and make decisions to navigate the environment

## Getting Started with Machine Learning
--------------------------------------

To get started with machine learning, you'll need:

1.  A data set relevant to your problem (e.g., images, text, sensor readings)
2.  Python or a similar programming language
3.  ML.NET (Microsoft's open-source machine learning platform)

Here's an example of how to train a simple classification model using ML.NET:
```csharp
using Microsoft.ML;
using Microsoft.ML.Data;

class TrainModel
{
    public static void Main(string[] args)
    {
        // Define the data set
        var data = new[]
        {
            new { Image = "image1.jpg", Label = "dog" },
            new { Image = "image2.jpg", Label = "cat" },
            new { Image = "image3.jpg", Label = "bird" }
        };

        // Load the ML.NET model
        var pipeline = new TrainingContext();
        pipeline.DataView = data;
        pipeline.Model = new ModelBuilder()
            .AddDataSets("Image")
            .Build();

        // Train the model
        pipeline.TrainAsync().Wait();

        // Make predictions on a new image
        var prediction = pipeline.Predict(new Image { Image = "image4.jpg" });
        Console.WriteLine($"Prediction: {prediction.Label}");
    }
}
```
In this example, we define a simple classification problem with two classes (dogs and cats) and train a model using the `TrainingContext` class. We then use the trained model to make predictions on a new image.

## Conclusion
----------

Machine learning is a powerful tool that has numerous applications in various fields. By understanding the basics of machine learning, including classification and regression, and applying them to real-world problems, you can unlock the potential of ML technology. With ML.NET as an open-source platform, you can start building your own machine learning models today.

## Practical Application
----------------------

### Image Classification

*   Use a dataset of images with their respective objects (e.g., dogs, cats, buildings)
*   Train a model using the `TrainingContext` class
*   Make predictions on new images using the trained model

Example:
```python
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Load the dataset
dataset = datasets.load_digits()

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(dataset.data, dataset.target, test_size=0.2)

# Create a Keras model
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(8, 8, 1)))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dense(len(dataset.target_names), activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10)

# Make predictions on new data
predictions = model.predict(X_test)
```
In this example, we use a dataset of images with their respective objects and train a Keras model using Convolutional Neural Networks (CNNs). We then make predictions on new images using the trained model.


──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_quiz_req
──────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Type  'quiz'  to test your knowledge.
  Type  'more'  to re-read the content.
────────────────────────────────────────────────────────────
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.
  Please type 'quiz' or 'more'.


## 10. Inspect Session — Roadmap & Resources

Run this cell after the LearnFlow graph finishes to display the full roadmap and all collected resources.

In [ ]:
def print_roadmap(state):
    """Render the roadmap table from any LearnFlowState."""
    if not state.roadmap:
        print("No roadmap yet — run the LearnFlow graph first.")
        return

    level_badge = {"beginner": "🟢 Beginner", "intermediate": "🟡 Intermediate",
                   "advanced": "🔴 Advanced"}.get(state.level, state.level or "—")

    lines = [
        f"## 📚 Learning Roadmap — *{state.topic}*",
        f"",
        f"**Level:** {level_badge}",
        f"",
        f"**Knowledge gaps:**",
    ]
    for g in state.gaps:
        lines.append(f"- {g}")
    lines += [
        f"",
        f"| # | Module | Objective | Concepts | Duration |",
        f"|---|--------|-----------|----------|----------|",
    ]
    for m in state.roadmap:
        concepts = ", ".join(m.get("concepts", []))
        lines.append(
            f"| {m.get('module','?')} | **{m.get('title','?')}** "
            f"| {m.get('objective','')} | {concepts} | ⏱ {m.get('duration','?')} |"
        )
    total_min = 0
    for m in state.roadmap:
        d = m.get("duration", "")
        nums = re.findall(r"\d+", d)
        if nums:
            mins = int(nums[0]) * (60 if "hour" in d else 1)
            total_min += mins
    total_str = f"{total_min // 60}h {total_min % 60:02d}min" if total_min >= 60 else f"{total_min} min"
    lines.append(f"")
    lines.append(f"*{len(state.roadmap)} modules · estimated total: {total_str}*")
    display(Markdown("\n".join(lines)))


def print_resources(state):
    """Render the collected web resources table from any LearnFlowState."""
    if not state.resources:
        print("No resources yet — run the LearnFlow graph first.")
        return

    lines = [
        f"## 🔍 Collected Resources ({len(state.resources)})",
        f"",
        f"| # | Title | Snippet |",
        f"|---|-------|---------|",
    ]
    for i, r in enumerate(state.resources, 1):
        title   = f"[{r['title'][:55]}]({r['url']})" if r.get("url") else r.get("title", "")[:55]
        snippet = r.get("snippet", "")[:100].replace("|", "\\|")
        lines.append(f"| {i} | {title} | {snippet}… |")
    display(Markdown("\n".join(lines)))


# ── Run ───────────────────────────────────────────────────────────────────────
# Uses `final_state` produced by cell 43 (the LearnFlow run cell).
# Re-run this cell any time to re-render without re-running the whole graph.
try:
    print_roadmap(final_state)
    print_resources(final_state)
except NameError:
    # final_state not set yet — auto-run the graph now
    print("⚡ `final_state` not found. Starting LearnFlow automatically...")
    _topic = LF_TOPIC if "LF_TOPIC" in dir() else "Machine Learning"
    graph = build_learnflow_graph()
    final_state = graph.run(LearnFlowState(topic=_topic))
    print_roadmap(final_state)
    print_resources(final_state)


⚡ `final_state` not found. Starting LearnFlow automatically...

──────────────────────────────────────────────────
[GRAPH] ▶  generate_assessment
──────────────────────────────────────────────────
[1/4] Generating diagnostic assessment for: Machine Learning
  Generated 8 questions.

──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_assessment
──────────────────────────────────────────────────

📋  DIAGNOSTIC ASSESSMENT — Machine Learning

Q1: What is the main difference between supervised and unsupervised learning?
   Machine Learning is an algorithm
   Machine Learning involves a model to predict
   Machine Learning algorithms are usually used for classification
   Machine Learning is based on data

Q2: What is the process of feature engineering in machine learning?
   Feature selection
   Feature extraction
   Dimensionality reduction
   Feature engineering

Q3: How do neural networks learn from data?
   By analyzing the input to create a model
   By training on l

## 🔍 Found 8 Learning Resources

| # | Title | Snippet |
|---|-------|---------|
| 1 | [FS# machine learning tutorial for beginners](https://jaxci.com/fs-reviews/machine-learning-tutorial-for-beginners/) | MachineLearningTutorialforBeginners\| Edureka » Feb 25, 2019 …MachineLearningMasters Progra… |
| 2 | [JX# machine learning tutorial for beginners](https://atonu.org/jx-reviews/machine-learning-tutorial-for-beginners/) | MachineLearningTutorialforBeginners\| Edureka » Feb 25, 2019 …MachineLearningMasters Progra… |
| 3 | [RH# machine learning tutorial for beginners](https://italianoit.com/rh-reviews/machine-learning-tutorial-for-beginners/) | MachineLearningTutorialforBeginners\| Edureka » Feb 25, 2019 …MachineLearningMasters Progra… |
| 4 | [Machine Learning tutorial - A beginner’s guide to Machi](https://nearlearn.com/blog/guide-for-begineers-machine-learning-technology/) | MachineLearningtutorial– Abeginner’s guide toMachineLearningTechnology ... Nearlearnis the… |
| 5 | [Machine Learning Tutorial - GeeksforGeeks](https://www.geeksforgeeks.org/machine-learning/machine-learning/) | Machinelearningis a branch of Artificial Intelligence that focuses on developing models an… |
| 6 | [PDFUndergraduate Fundamentals of Machine Learning](https://harvard-ml-courses.github.io/cs181-web-2024/static/cs181-textbook.pdf) | UndergraduateFundamentalsofMachineLearningThe initial version of this textbook was created… |
| 7 | [Introduction to Machine Learning Concepts - Training](https://learn.microsoft.com/en-us/training/modules/fundamentals-machine-learning/) | Machinelearningis the basis for most modern artificial intelligence solutions. A familiari… |
| 8 | [Machine Learning Fundamentals Handbook - Key Concepts, ](https://www.freecodecamp.org/news/machine-learning-handbook/) | Throughout this handbook, I'll include examples for eachMachineLearningalgorithm with its … |


──────────────────────────────────────────────────
[GRAPH] ▶  build_roadmap
──────────────────────────────────────────────────
[4/4] Building personalised roadmap...


## 📚 Your Personalised Learning Roadmap

**Topic:** Machine Learning  ·  **Level:** 🟢 Beginner

**Knowledge gaps addressed:**
- fundamentals of Machine Learning

---

| # | Module | Objective | Concepts | Duration |
|---|--------|-----------|----------|----------|
| 1 | **Machine Learning 101: Introduction** | Understand the basics of machine learning, including supervised and unsupervised learning, regression, classification, clustering, and dimensionality reduction. | c1, c2 | ⏱ 60 minutes |
| 2 | **Supervised Learning** | Learn about supervised learning algorithms such as linear regression, logistic regression, decision trees, and neural networks. | c1, c3 | ⏱ 90 minutes |
| 3 | **Unsupervised Learning** | Discover the basics of unsupervised learning techniques such as clustering, dimensionality reduction, and dimensionality exploration. | c2, c4 | ⏱ 90 minutes |
| 4 | **Deep Learning Fundamentals** | Learn the basics of deep learning concepts such as neural networks, convolutional neural networks (CNNs), and recurrent neural networks (RNNs). | c3, c5 | ⏱ 120 minutes |
| 5 | **Real-World Applications of Machine Learning** | Explore real-world applications of machine learning, including image classification, speech recognition, and natural language processing. | c2, c6 | ⏱ 90 minutes |
| 6 | **Practical Implementation** | Practice implementing machine learning algorithms using Python and popular libraries such as scikit-learn, TensorFlow, and PyTorch. | c4, c7 | ⏱ 120 minutes |

---
*6 modules · estimated total: 9h 30min*


──────────────────────────────────────────────────
[GRAPH] ▶  generate_content
──────────────────────────────────────────────────
Generating content — Module 1/6: Machine Learning 101: Introduction

📖  MODULE 1: Machine Learning 101: Introduction


...

- **[Machine Learning | Coursera](https://www.coursera.org/specializations/machine-learning-introduction)**  
  It provides a broad introduction to modern machine learning, including supervised learning (multiple

- **[Introduction to Machine Learning | Coursera](https://www.coursera.org/learn/machine-learning-duke)**  
  June 27, 2021 -This courseprovides a foundational understanding of machine learning models (logistic

- **[Machine Learning Introduction for Everyone | Coursera](https://www.coursera.org/learn/machine-learning-introduction-for-everyone)**  
  November 1, 2022 -This three-module course introduces machine learning and data science for everyone

- **[Introduction to Machine Learning | Electrical Engineering and Computer Science | MIT OpenCourseWare](https://ocw.mit.edu/courses/6-036-introduction-to-machine-learning-fall-2020/)**  
  This courseintroduces principles, algorithms, and applications of machine learning from the point of

- **[Introduction to Machine Learning | MIT Open Learning Library](https://openlearninglibrary.mit.edu/courses/course-v1:MITx+6.036+1T2019/about)**  
  This courseintroduces principles, algorithms, and applications of machine learning from the point of

- **[Lecture 11: Introduction to Machine Learning | Introduction to Computational Thinking and Data Science | Electrical Engineering and Computer Science | MIT OpenCourseWare](https://ocw.mit.edu/courses/6-0002-introduction-to-computational-thinking-and-data-science-fall-2016/resources/lecture-11-introduction-to-machine-learning/)**  
  Description: In this lecture, Prof.Guttag introduces machine learning and shows examples of supervis


💡  Review these before retrying — they cover the gaps detected in your quiz.
────────────────────────────────────────────────────────────

  ❌  Score 50 < 70. Reviewing content again…

──────────────────────────────────────────────────
[GRAPH] ▶  generate_content
──────────────────────────────────────────────────
Generating content — Module 1/6: Machine Learning 101: Introduction

📖  MODULE 1: Machine Learning 101: Introduction


**Module: Machine Learning 101: Introduction**

### Overview

Machine learning is a subset of artificial intelligence (AI) that enables computers to learn from data without being explicitly programmed for every task. It's a crucial concept in the field of computer science and has numerous applications across various industries.

### Key Concepts

#### Supervised Learning

Supervised learning involves training a model on labeled data, where the input data is used as features to predict the target variable. In other words, you have a dataset with inputs (X) and outputs (Y), and your goal is to train a model that can map inputs to outputs.

**Example:**

Suppose you own an e-commerce website and want to build a recommendation system. You collect customer purchase history data along with product information. You label the data as "Recommended" or "Not Recommended." The machine learning algorithm trains on this labeled data, learns patterns in customer behavior, and becomes proficient at suggesting products that customers are likely to buy.

#### Unsupervised Learning

Unsupervised learning involves analyzing unlabeled data to discover patterns, relationships, or clusters. It's often used for dimensionality reduction, anomaly detection, and clustering.

**Example:**

Suppose you have a dataset of customer demographics (age, location, etc.) without any labels. You can use unsupervised learning techniques like k-means clustering to group customers by their demographic characteristics, identify patterns in the data, or detect anomalies.

#### Regression

Regression involves predicting continuous values based on input features. It's commonly used for forecasting sales, stock prices, or temperatures.

**Example:**

Suppose you want to predict the number of customers expected to buy a product based on its price. You collect historical sales data and use regression analysis to build a model that can forecast future sales.

#### Classification

Classification involves predicting categorical labels (e.g., spam vs. non-spam emails) based on input features. It's commonly used for image classification, text classification, or sentiment analysis.

**Example:**

Suppose you want to classify emails as spam or not spam based on their content. You collect labeled data where some emails are classified as spam and others as not spam. Your machine learning model learns to map input features (e.g., keyword patterns) to predicted labels.

#### Clustering

Clustering involves grouping similar data points into clusters based on input features. It's commonly used for customer segmentation, market basket analysis, or image clustering.

**Example:**

Suppose you have a dataset of customer purchasing behavior and want to segment customers by their purchase frequency and product category. Your machine learning model learns to group customers into clusters based on these features.

#### Dimensionality Reduction

Dimensionality reduction involves reducing the number of input features while preserving the most important information. It's commonly used for data visualization, feature selection, or dimensionality reduction.

**Example:**

Suppose you have a dataset with 10 features (e.g., age, location, etc.) and want to visualize only the top 3 most influential features. Your machine learning model reduces the number of input features while preserving the relationships between them.

### Practical Application

Machine learning has numerous practical applications in various industries:

* **Recommendation Systems:** Build recommendation systems that suggest products or services based on user behavior.
* **Image Classification:** Train image classification models to classify images into different categories (e.g., objects, scenes).
* **Speech Recognition:** Develop speech recognition systems that transcribe spoken words into text.
* **Natural Language Processing:** Build natural language processing models to analyze and generate human language.

### Summary

In this module, you've learned the basics of machine learning, including supervised and unsupervised learning, regression, classification, clustering, and dimensionality reduction. You've also applied these concepts to practical examples in various industries. Next, we'll dive deeper into each concept with more detailed explanations, examples, and code.

**Next Steps:**

1. Practice building simple machine learning models using popular libraries like Scikit-learn or TensorFlow.
2. Explore different algorithms and techniques for supervised and unsupervised learning.
3. Apply machine learning concepts to real-world problems in various industries.

### Additional Resources:

* **Edureka Tutorial:** Machine Learning Tutorial for Beginners
* **Machine Learning Masters Program:** https://www.edureka.co/masters/
* **GeeksforGeeks:** Machine Learning Tutorial


──────────────────────────────────────────────────
[GRAPH] ▶  wait_for_quiz_req
──────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Type  'quiz'  to test your knowledge.
  Type  'more'  to re-read the content.
────────────────────────────────────────────────────────────
  Please type 'quiz' or 'more'.


**Module: Machine Learning 101: Introduction**

### Overview

Machine learning is a subset of artificial intelligence (AI) that enables computers to learn from data without being explicitly programmed for every task. It's a crucial concept in the field of computer science and has numerous applications across various industries.

### Key Concepts

#### Supervised Learning

Supervised learning involves training a model on labeled data, where the input data is used as features to predict the target variable. In other words, you have a dataset with inputs (X) and outputs (Y), and your goal is to train a model that can map inputs to outputs.

**Example:**

Suppose you own an e-commerce website and want to build a recommendation system. You collect customer purchase history data along with product information. You label the data as "Recommended" or "Not Recommended." The machine learning algorithm trains on this labeled data, learns patterns in customer behavior, and becomes proficient at suggesting products that customers are likely to buy.

#### Unsupervised Learning

Unsupervised learning involves analyzing unlabeled data to discover patterns, relationships, or clusters. It's often used for dimensionality reduction, anomaly detection, and clustering.

**Example:**

Suppose you have a dataset of customer demographics (age, location, etc.) without any labels. You can use unsupervised learning techniques like k-means clustering to group customers by their demographic characteristics, identify patterns in the data, or detect anomalies.

#### Regression

Regression involves predicting continuous values based on input features. It's commonly used for forecasting sales, stock prices, or temperatures.

**Example:**

Suppose you want to predict the number of customers expected to buy a product based on its price. You collect historical sales data and use regression analysis to build a model that can forecast future sales.

#### Classification

Classification involves predicting categorical labels (e.g., spam vs. non-spam emails) based on input features. It's commonly used for image classification, text classification, or sentiment analysis.

**Example:**

Suppose you want to classify emails as spam or not spam based on their content. You collect labeled data where some emails are classified as spam and others as not spam. Your machine learning model learns to map input features (e.g., keyword patterns) to predicted labels.

#### Clustering

Clustering involves grouping similar data points into clusters based on input features. It's commonly used for customer segmentation, market basket analysis, or image clustering.

**Example:**

Suppose you have a dataset of customer purchasing behavior and want to segment customers by their purchase frequency and product category. Your machine learning model learns to group customers into clusters based on these features.

#### Dimensionality Reduction

Dimensionality reduction involves reducing the number of input features while preserving the most important information. It's commonly used for data visualization, feature selection, or dimensionality reduction.

**Example:**

Suppose you have a dataset with 10 features (e.g., age, location, etc.) and want to visualize only the top 3 most influential features. Your machine learning model reduces the number of input features while preserving the relationships between them.

### Practical Application

Machine learning has numerous practical applications in various industries:

* **Recommendation Systems:** Build recommendation systems that suggest products or services based on user behavior.
* **Image Classification:** Train image classification models to classify images into different categories (e.g., objects, scenes).
* **Speech Recognition:** Develop speech recognition systems that transcribe spoken words into text.
* **Natural Language Processing:** Build natural language processing models to analyze and generate human language.

### Summary

In this module, you've learned the basics of machine learning, including supervised and unsupervised learning, regression, classification, clustering, and dimensionality reduction. You've also applied these concepts to practical examples in various industries. Next, we'll dive deeper into each concept with more detailed explanations, examples, and code.

**Next Steps:**

1. Practice building simple machine learning models using popular libraries like Scikit-learn or TensorFlow.
2. Explore different algorithms and techniques for supervised and unsupervised learning.
3. Apply machine learning concepts to real-world problems in various industries.

### Additional Resources:

* **Edureka Tutorial:** Machine Learning Tutorial for Beginners
* **Machine Learning Masters Program:** https://www.edureka.co/masters/
* **GeeksforGeeks:** Machine Learning Tutorial